# System Setup

## Imports and Installs

### Installs

In [ ]:
!pip install geopandas
!pip install geojson
!pip install overpass
!pip install alphashape

### Imports

In [ ]:
import pandas as pd
import json
import geojson
#https://github.com/mvexel/overpass-api-python-wrapper/blob/main/README.md
import overpass
#https://pypi.org/project/osm2geojson/
import csv
import os
import logging
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from shapely.geometry import Point, Polygon
import requests
import time
from google.colab import drive
import sys
import ast
from alphashape import alphashape
import geopandas as gpd
import os
from collections import defaultdict
import os.path
from tqdm import tqdm

In [ ]:
import numpy as np
import pickle
import random
import json
from itertools import permutations
import time as time_module
import networkx as nx
from datetime import datetime
import geopy.distance
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

## Parameters

In [ ]:
state_name = "Ohio"
city_name = "Columbus"
mini = True

## Connecting to Drive

We first mount the drive. This step requires permission to be given. Running the cell should open a permission page to allow colab access to google drive.

In [ ]:
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


Setting the file directory to the data folder. All files will be referenced to this working directory.

Folder structure:

```
📁Drive/
└─ 📁<personal_dir>/
   ├─ 📁<city_name_1> - RL Delivery Data/
   ├─ 📁<city_name_2> - RL Delivery Data/
   └─ 📁Raw GeoJsons/
```


## Assign Data directory path

In [ ]:
personal_dir = "./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/"
data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"
output_dir = data_dir + "/UMST Graph/bundling"


In [ ]:
try:
  os.mkdir(data_dir)
except:
  print(data_dir + " Already Exists")

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

# See directory content
data = os.listdir(data_dir)
print("Files in directory : ", data)

./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data Already Exists
Output directory: ./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data/UMST Graph/bundling
Files in directory :  ['Census Data', 'Images', 'Hotspot Data', 'Original Location Data', 'Processed Location Data', 'Q Tables', 'UMST Graph', 'gh_cache', 'Deliveries', 'avg_hotspot_data.json', 'experiment_results']


## Load Data

#### Loading city data

In [ ]:
census_df = gpd.read_file(data_dir + "/Census Data/census_tract_data.geojson")
num_hotspots = len(census_df.index)
print(num_hotspots)

# === FILE PATHS ===
umst_path = data_dir + "/UMST Graph/graphs/umst_graph.graphml"
mst_path = data_dir + "/UMST Graph/graphs/mst_graph.graphml"
hotspot_path = data_dir + "/UMST Graph/graphs/gh_hotspot_graph.graphml"
census_path = data_dir + "/Census Data/census_tract_data.geojson"
hotspot_geojson_path = data_dir + "/Hotspot Data/hotspot_data.geojson"

# === 1. LOAD GRAPHS ===
print("\n[1] Loading graphs from files...")
umst_graph = nx.read_graphml(umst_path)
mst_graph = nx.read_graphml(mst_path)
hotspot_graph = nx.read_graphml(hotspot_path)

# Load census tract boundaries for visualization
try:
    census_gdf = gpd.read_file(census_path, crs='EPSG:4326')
    print(f"    ✓ Census tracts loaded: {len(census_gdf)} tracts")
except Exception as e:
    print(f"    ⚠ Warning: Could not load census data: {e}")
    print(f"    Continuing without census tract boundaries...")
    census_gdf = None

print(f"    ✓ Hotspot Graph: {hotspot_graph.number_of_nodes()} nodes, {hotspot_graph.number_of_edges()} edges")
print(f"    ✓ MST Graph: {mst_graph.number_of_nodes()} nodes, {mst_graph.number_of_edges()} edges")
print(f"    ✓ UMST Graph: {umst_graph.number_of_nodes()} nodes, {umst_graph.number_of_edges()} edges")


# Get number of hotspots (nodes)
num_hotspots = umst_graph.number_of_nodes()
print(f"Number of hotspots: {num_hotspots}")

26

[1] Loading graphs from files...
    ✓ Census tracts loaded: 26 tracts
    ✓ Hotspot Graph: 26 nodes, 325 edges
    ✓ MST Graph: 26 nodes, 25 edges
    ✓ UMST Graph: 26 nodes, 47 edges
Number of hotspots: 26


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option CRS
  return ogr_read(


In [ ]:
def create_matrices_from_graph(graph: nx.Graph,
                               node_to_id: dict,
                               dist_attr: str = 'distance',
                               time_attr: str = 'time',
                               as_dataframe: bool = True,
                               calculate_shortest_paths: bool = True):
    n = len(node_to_id)

    dist_matrix = np.full((n, n), np.inf, dtype=float)
    time_matrix = np.full((n, n), np.inf, dtype=float)
    np.fill_diagonal(dist_matrix, 0)
    np.fill_diagonal(time_matrix, 0)

    edge_count = 0
    for u, v, data in graph.edges(data=True):
        i, j = node_to_id[u], node_to_id[v]
        dist = float(data.get(dist_attr, 0))
        time = float(data.get(time_attr, 0))

        time *= 60 # COnvert minutes into seconds

        dist_matrix[i, j] = dist
        dist_matrix[j, i] = dist
        time_matrix[i, j] = time
        time_matrix[j, i] = time
        edge_count += 1

    print(f"Graph: {n} nodes, {edge_count} edges")

    if calculate_shortest_paths:
        print("Calculating shortest paths...")
        dist_matrix = _compute_shortest_paths(dist_matrix)
        time_matrix = _compute_shortest_paths(time_matrix)

        unreachable = np.sum(dist_matrix == np.inf) - n
        total_pairs = n * (n - 1)
        print(f"Reachable: {total_pairs - unreachable}/{total_pairs} pairs")
        if unreachable > 0:
            print(f"WARNING: {unreachable} pairs unreachable")

    if as_dataframe:
        node_labels = list(node_to_id.keys())
        dist_matrix = pd.DataFrame(dist_matrix, index=node_labels, columns=node_labels)
        time_matrix = pd.DataFrame(time_matrix, index=node_labels, columns=node_labels)

    return dist_matrix, time_matrix


def _compute_shortest_paths(matrix):
    matrix_sparse = matrix.copy()
    matrix_sparse[matrix_sparse == np.inf] = 0
    sparse_matrix = csr_matrix(matrix_sparse)
    shortest_paths = dijkstra(csgraph=sparse_matrix, directed=False, return_predecessors=False)
    return shortest_paths

In [ ]:
nodes = sorted(umst_graph.nodes())
index_to_tract = {idx: node for idx, node in enumerate(nodes)}  # id → node
tract_to_index = {node: idx for idx, node in enumerate(nodes)}  # node → id

# Get numpy arrays with integer indices
full_dist_matrix, full_time_matrix = create_matrices_from_graph(
    graph=umst_graph,
    node_to_id=tract_to_index,  # Use tract_to_index (node → id)
    calculate_shortest_paths=True,
    as_dataframe=False  # ← Important! Get numpy arrays, not DataFrames
)

print("Distance matrix shape:", full_dist_matrix.shape)
print("Time matrix shape:", full_time_matrix.shape)
print("\nSample distance matrix (integer indexed):")
print(full_dist_matrix[:10, :10])
print("\nSample time matrix (integer indexed):")
print(full_time_matrix[:10, :10])

# Now you can access by integer index
print(f"\nDistance from node 0 to node 5: {full_dist_matrix[0, 5]:.3f} km")
print(f"Time from node 0 to node 5: {full_time_matrix[0, 5]:.0f} seconds")

Graph: 26 nodes, 47 edges
Calculating shortest paths...
Reachable: 676/650 pairs
Distance matrix shape: (26, 26)
Time matrix shape: (26, 26)

Sample distance matrix (integer indexed):
[[0.    1.016 1.829 0.967 1.842 4.296 5.003 2.928 2.861 3.622]
 [1.016 0.    1.711 1.929 2.804 5.258 5.965 3.89  3.823 3.504]
 [1.829 1.711 0.    1.92  1.464 3.918 4.625 2.55  2.483 1.793]
 [0.967 1.929 1.92  0.    0.875 3.329 4.036 1.961 1.894 3.106]
 [1.842 2.804 1.464 0.875 0.    2.454 3.161 1.086 1.019 2.231]
 [4.296 5.258 3.918 3.329 2.454 0.    0.707 2.843 3.473 4.685]
 [5.003 5.965 4.625 4.036 3.161 0.707 0.    2.415 3.154 4.366]
 [2.928 3.89  2.55  1.961 1.086 2.843 2.415 0.    0.739 1.951]
 [2.861 3.823 2.483 1.894 1.019 3.473 3.154 0.739 0.    1.212]
 [3.622 3.504 1.793 3.106 2.231 4.685 4.366 1.951 1.212 0.   ]]

Sample time matrix (integer indexed):
[[  0. 207. 299. 156. 273. 601. 758. 429. 377. 545.]
 [207.   0. 234. 258. 375. 703. 860. 531. 479. 569.]
 [299. 234.   0. 289. 261. 589. 746. 417

In [ ]:
# print("Graph edges with attributes:")
# for u, v, data in umst_graph.edges(data=True):
#     print(f"{u} -- {v} | distance={data.get('distance')} | time={data.get('time')}")


#### Creating Full/"Estimate" Distance and Time matrices
**Note:** The Estimate matrices will only be created if they do not already exist.

In [ ]:
if "estimate_distance_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data") or "estimate_time_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data"):
  # Import hotspots and order according to tract_to_index
  hotspot_df = gpd.read_file(data_dir + "/Hotspot Data/hotspot_data.geojson")
  hotspot_df['index_order'] = hotspot_df['GEOID'].map(tract_to_index)
  hotspot_df = hotspot_df.sort_values(by='index_order').reset_index(drop=True)
  hotspot_df = hotspot_df.drop(columns=['index_order'])


  # Turn hotspot dataframe into a matrix of coordinates getting the distance between them
  coords = hotspot_df['geometry'].apply(lambda geom: (geom.x, geom.y)).tolist()
  estimate_dist_matrix = np.zeros((len(coords), len(coords)))

  # Calculate pairwise geodesic distances
  for i, coord1 in enumerate(coords):
    for j, coord2 in enumerate(coords):
      estimate_dist_matrix[i, j] = geodesic(coord1, coord2).meters if i != j else 0.0

  # Add the average distance longer that actual roads take to the crow-flies distances
  dist_diff_matrix = full_dist_matrix - estimate_dist_matrix
  dist_diff_matrix = dist_diff_matrix[dist_diff_matrix < 100000000]
  avg_increase = dist_diff_matrix.mean()
  estimate_dist_matrix = estimate_dist_matrix + avg_increase

  # Get the speed of the distances from the distance/time matrices
  speed_matrix = full_dist_matrix / full_time_matrix
  np.nan_to_num(speed_matrix, 0)  # Set nan values to 0

  # Exclude values where speed is 1, as the vast majority (if not all) indicate edges that dont exist
  valid_speeds_mask = (speed_matrix != 1) & ~np.isnan(speed_matrix)

  # Mask speed matrix
  valid_speeds = speed_matrix[valid_speeds_mask]
  average_speed = np.mean(valid_speeds) # Get the mean of the matrix, this gives the average speed for each distance travelled

  estimate_time_matrix = estimate_dist_matrix / average_speed  # Use this and the full distance matrix to create the full time matrix

  np.save(data_dir + f"/Hotspot Data/estimate_distance_adjacency_matrix.npy", estimate_dist_matrix)
  np.save(data_dir + f"/Hotspot Data/estimate_time_adjacency_matrix.npy", estimate_time_matrix)

estimate_dist_matrix = np.load(data_dir + f"/Hotspot Data/estimate_distance_adjacency_matrix.npy")
estimate_time_matrix = np.load(data_dir + f"/Hotspot Data/estimate_time_adjacency_matrix.npy")

Below the full distance/time matrices are created and updated as necessary

In [ ]:
if "full_distance_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data") or "full_time_adjacency_matrix.npy" not in os.listdir(data_dir + "/Hotspot Data"):
  print("Creating new")

  np.save(data_dir + f"/Hotspot Data/full_distance_adjacency_matrix.npy", full_dist_matrix)
  np.save(data_dir + f"/Hotspot Data/full_time_adjacency_matrix.npy", full_time_matrix)

else:
  print("Already exist")
  full_dist_matrix = np.load(data_dir + f"/Hotspot Data/full_distance_adjacency_matrix.npy")
  full_time_matrix = np.load(data_dir + f"/Hotspot Data/full_time_adjacency_matrix.npy")

  # Copy distance/time_matrix data onto full, and save it again.
  for i in range(len(census_df.index)):
    for j in range(len(census_df.index)):
      if full_dist_matrix[i, j] == sys.maxsize and full_dist_matrix[i, j] != sys.maxsize:
        full_dist_matrix[i, j] = full_dist_matrix[i, j]
        full_time_matrix[i, j] = full_time_matrix[i, j]

  np.save(data_dir + f"/Hotspot Data/full_distance_adjacency_matrix.npy", full_dist_matrix)
  np.save(data_dir + f"/Hotspot Data/full_time_adjacency_matrix.npy", full_time_matrix)

Already exist



# Simulation Materials

#### Global Variables and Functions

In [ ]:
INTERVALS = [900, 1200, 1500, 1800, 2700, 3600, 5400, 7200, 10800, 14400]

In [ ]:
# Changed the directory
deliveries_dir = data_dir + f"/Deliveries"
try:
  os.mkdir(deliveries_dir)
except:
  print(deliveries_dir + " Already Exists")

./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data/Deliveries Already Exists


### Deliveries

#### Delivery Class

In [ ]:
#### Delivery Class ####

class Delivery:
    """
    Represents a single delivery in the UMST graph system.
    """
    id = 0

    def __init__(self, start_node, end_node, start_time, percent_max_dist=100):
        # IMMUTABLE
        self.id = Delivery.id
        Delivery.id += 1
        self.start_node = start_node
        self.end_node = end_node
        self.start_time = start_time
        self.percent_max_dist = percent_max_dist

        # MUTABLE
        self.current_node = self.start_node
        self.time_till_next_node = 0
        self.end_time = None
        self.in_transition = False
        self.is_primary = True  # Primary delivery in a bundle pays vehicle cost
        self.next_node = None
        self.completed = False
        self.successful = False
        self.num_car_changes = 2  # Initial PDC->node + final node->PDC
        self.bundle = None  # Reference to DeliveryBundle if bundled
        self.bundle_id = None  # ID of the bundle this delivery belongs to
        self.path = [self.start_node]
        self.distance_traveled = 0
        self.time_limit = self.calculate_time_limit()

        # DEBUG/TRACKING VARIABLES
        self.did_bundle = False
        self.bundled_with = []  # List of delivery IDs bundled with
        self.bundle_start_node = None  # Node where bundling started

    def calculate_time_limit(self):
        """Calculate time limit based on direct travel time."""
        global INTERVALS, full_time_matrix

        path_time = full_time_matrix[self.start_node, self.end_node]

        for interval in INTERVALS:
            if path_time * 1.5 <= interval:
                return interval
        return float('inf')

    def check_bundling_feasibility(self, other_delivery, max_detour_percent):
      """
      Check if bundling with another delivery is feasible.

      Args:
          other_delivery: Another Delivery object
          max_detour_percent: Maximum allowed detour percentage

      Returns:
          bool: True if bundling is feasible
      """
      global full_dist_matrix

      # print(f"Check_bundling_feasibility: self: {self.id, self.start_time}, other_delivery:{other_delivery.id, other_delivery.start_time}, max_detour_percent:{max_detour_percent}")

      # Can't bundle if either is completed
      if self.completed or other_delivery.completed:
          print(f"Return False - one delivery already completed")
          return False

      # SAME DESTINATION = PERFECT FOR BUNDLING! (Zero detour)
      if self.end_node == other_delivery.end_node:
          print(f"Return True - same destination (PERFECT bundling, zero detour!)")
          return True

      # Check if percent_max_dist allows bundling (100 means no bundling allowed)
      if self.percent_max_dist == 100:
          print(f"Return False - percent_max_dist is 100 (no bundling)")
          return False

      # Current positions (should be the same when checking bundling)
      node1 = self.next_node if self.in_transition else self.current_node
      node2 = other_delivery.next_node if other_delivery.in_transition else other_delivery.current_node

      # Get destinations (we know they're different at this point)
      dest1 = self.end_node
      dest2 = other_delivery.end_node

      # Calculate distance between destinations
      distance_between_destinations = full_dist_matrix[dest1, dest2]

      # Calculate direct distances from current position to each destination
      direct_dist_1 = full_dist_matrix[node1, dest1]
      direct_dist_2 = full_dist_matrix[node2, dest2]

      # Maximum distance in the network (for normalization)
      max_distance = np.max(full_dist_matrix[full_dist_matrix != sys.maxsize])

      # print(f"Current nodes: {node1}, {node2}")
      # print(f"Destinations: {dest1}, {dest2}")
      # print(f"Distance between destinations: {distance_between_destinations}")
      # print(f"Direct distance to dest1: {direct_dist_1}")
      # print(f"Direct distance to dest2: {direct_dist_2}")
      # print(f"Max distance in network: {max_distance}")

      # Check 1: Are destinations close enough relative to max distance?
      normalized_dest_distance = distance_between_destinations / max_distance
      if normalized_dest_distance > self.percent_max_dist / 100:
          print(f"Return False - destinations too far apart ({normalized_dest_distance:.2%} > {self.percent_max_dist}%)")
          return False

      # Check 2: Calculate detour for bundled route
      # Option 1: Go to dest1 first, then dest2
      bundled_dist_option1 = direct_dist_1 + distance_between_destinations
      detour_option1 = bundled_dist_option1 - direct_dist_2

      # Option 2: Go to dest2 first, then dest1
      bundled_dist_option2 = direct_dist_2 + distance_between_destinations
      detour_option2 = bundled_dist_option2 - direct_dist_1

      # Use the better option (less detour)
      min_detour = min(detour_option1, detour_option2)
      original_distance = max(direct_dist_1, direct_dist_2)

      if original_distance > 0:
          detour_percent = (min_detour / original_distance) * 100
      else:
          detour_percent = 0

      # print(f"Bundled option 1 (dest1->dest2): {bundled_dist_option1:.2f}, detour: {detour_option1:.2f}")
      # print(f"Bundled option 2 (dest2->dest1): {bundled_dist_option2:.2f}, detour: {detour_option2:.2f}")
      # print(f"Min detour: {min_detour:.2f}, detour percentage: {detour_percent:.1f}%")

      # Check if detour is acceptable
      if detour_percent > max_detour_percent:
          print(f"Return False - detour too large ({detour_percent:.1f}% > {max_detour_percent}%)")
          return False

      print(f"Return True - bundling feasible")
      return True


    def reset(self, p=-1):
        """Reset delivery to initial state."""
        percent = self.percent_max_dist if p < 0 else p

        self.current_node = self.start_node
        self.time_till_next_node = 0
        self.end_time = None
        self.in_transition = False
        self.is_primary = True
        self.next_node = None
        self.completed = False
        self.successful = False
        self.num_car_changes = 2
        self.bundle = None
        self.bundle_id = None
        self.path = [self.start_node]
        self.distance_traveled = 0
        self.percent_max_dist = percent
        self.time_limit = self.calculate_time_limit()

        # DEBUG
        self.did_bundle = False
        self.bundled_with = []
        self.bundle_start_node = None

    def __lt__(self, other):
        """Comparison for sorting."""
        if self.start_time != other.start_time:
            return self.start_time < other.start_time
        elif len(self.path) != len(other.path):
            return len(self.path) < len(other.path)
        else:
            return self.id < other.id

    def __str__(self):
        return f"Delivery {self.id}\n" \
               f"Start Time: \t {self.start_time}\n" \
               f"Start Node: \t {self.start_node}\n" \
               f"End Node: \t {self.end_node}\n" \
               f"In Transition: \t {self.in_transition}\n" \
               f"Time limit: \t {self.time_limit}\n" \
               f"Current Node: \t {self.current_node}\n" \
               f"Next Node: \t {self.next_node}\n" \
               f"Bundle ID: \t {self.bundle_id}\n" \
               f"Bundled With: \t {self.bundled_with}\n" \
               f"Path: \t \t {self.path}\n" \
               f"Completed: \t {self.completed}\n"

#### Delivery Bundle

In [ ]:
#### DeliveryBundle Class ####

class DeliveryBundle:
    """
    Represents a bundle of multiple deliveries sharing a route.
    Handles routing and vehicle management for bundled deliveries.
    """
    bundle_id_counter = 0

    def __init__(self, deliveries, distance_matrix, time_matrix):
        self.id = DeliveryBundle.bundle_id_counter
        DeliveryBundle.bundle_id_counter += 1

        self.deliveries = list(deliveries)  # List of Delivery objects
        self.distance_matrix = distance_matrix
        self.time_matrix = time_matrix

        # Set primary delivery (first one)
        self.primary_delivery = self.deliveries[0]
        self.primary_delivery.is_primary = True
        for d in self.deliveries[1:]:
            d.is_primary = False

        # Calculate optimal route through all destinations
        self.route = self.calculate_bundle_route()
        self.current_route_index = 0
        self.active = True

        # Link deliveries to this bundle
        for d in self.deliveries:
            d.bundle = self
            d.bundle_id = self.id
            d.did_bundle = True
            d.bundled_with = [delivery.id for delivery in self.deliveries if delivery.id != d.id]

    def calculate_bundle_route(self):
        """
        Calculate optimal route visiting all delivery destinations.
        Uses nearest neighbor heuristic for efficiency.

        Returns:
            list: Ordered list of nodes to visit
        """
        # Start from the current node of primary delivery
        start = self.primary_delivery.current_node
        destinations = [d.end_node for d in self.deliveries]

        # Nearest neighbor algorithm
        route = [start]
        unvisited = set(destinations)
        current = start

        while unvisited:
            # Find nearest unvisited destination
            nearest = min(unvisited, key=lambda x: self.distance_matrix[current, x])
            route.append(nearest)
            current = nearest
            unvisited.remove(nearest)

        return route

    def get_next_node(self):
        """Get next node in the bundle route."""
        if self.current_route_index < len(self.route) - 1:
            return self.route[self.current_route_index + 1]
        return None

    def advance_route(self):
        """Move to next node in route."""
        self.current_route_index += 1

    def remove_completed_deliveries(self):
        """Remove deliveries that have reached their destination."""
        self.deliveries = [d for d in self.deliveries if not d.completed]

        if not self.deliveries:
            self.active = False
        elif self.primary_delivery.completed:
            # Assign new primary
            self.primary_delivery = self.deliveries[0]
            self.primary_delivery.is_primary = True

    def __str__(self):
        return f"Bundle {self.id}: {len(self.deliveries)} deliveries, " \
               f"Route: {self.route}, Active: {self.active}"


#### DeliveryList Class

In [ ]:
#### DeliveryList Class ####

class DeliveryList:
    """
    Manages collection of deliveries with distribution generation.
    Simplified for UMST - no distance-based filtering needed.
    """

    def __init__(self, distance_based:str, load:int, percent_max_dist:float,
                 hours:int=1, peaks:list=[0.25, 0.75], sigma:float=10,
                 gen_multi:bool=False, use_generated_paths:bool=True, multi_id:int=0):
        self.distance_based = distance_based
        self.load = load
        self.hours = hours
        self.peaks = peaks
        self.sigma = sigma
        self.percent_max_dist = percent_max_dist
        self.gen_multi = gen_multi
        self.multi_id = multi_id

        print("Generating ID...")
        self.id = DeliveryList.gen_deliveries_id(
            distance_based=self.distance_based,
            load=self.load,
            percent_max_dist=self.percent_max_dist,
            hours=self.hours,
            peaks=self.peaks,
            sigma=self.sigma,
            gen_multi=self.gen_multi,
            multi_id=self.multi_id
        )

        print("done.\n\nGenerating distribution...")
        self.distribution, self.loads = self.generate_distribution()

        print("done.\n\nGenerating deliveries...")
        self.deliveries = self.generate_deliveries(use_generated_paths)

        print("\ndone.\n\nSaving deliveries...")
        DeliveryList.save_deliveries(self)
        print("done.")

    @staticmethod
    def gen_deliveries_id(distance_based:str, load:int, percent_max_dist:float,
                         hours:int=1, peaks:list=[0.25, 0.75], sigma:float=10,
                         gen_multi:bool=False, multi_id:int=0):
        """Generate unique ID for delivery list."""
        global deliveries_dir

        if not gen_multi and multi_id > 0:
            raise ValueError("gen_multi must be True if multi_id is specified.")

        # Determine distance-based code
        base_code_map = {
            "further": "1",
            "closer": "2",
            "one_cluster": "3"
        }
        base_code = base_code_map.get(distance_based, "0")

        # Construct ID string
        id_str = (
            f"{base_code}"
            f"-{load:04d}"
            f"-{hours:03d}"
            f"-{sigma:03d}"
            f"-{peaks}"
            f"-{percent_max_dist:03d}"
        )

        # Handle multi-generation
        list_matching_ids = [f for f in os.listdir(deliveries_dir)
                           if f.startswith(id_str)] if gen_multi else []

        final_id = id_str

        if multi_id > 0:
            final_multi_id = multi_id - 1
        elif list_matching_ids:
            multi_ids = (
                int(f.split('-')[-1])
                for f in list_matching_ids
                if '[' not in f.split('-')[-2] and f.split('-')[-1].isdigit()
            )
            final_multi_id = max(multi_ids, default=0)
        else:
            return final_id

        final_id += f'-{final_multi_id + 1:02d}' if final_multi_id else '-01'
        return final_id

    def generate_distribution(self):
        """Generate temporal distribution of deliveries."""
        mu_list = [peak * 60 * self.hours for peak in self.peaks]
        x = np.linspace(0, 60 * self.hours, 60 * self.hours)

        y_combined = np.zeros_like(x)
        for mu in mu_list:
            y_combined += (1 / (self.sigma * np.sqrt(2 * np.pi))) * \
                         np.exp(-0.5 * ((x - mu) / self.sigma) ** 2)

        loads = [int((self.load * (self.sigma * np.sqrt(2 * np.pi))) * y)
                for y in y_combined]

        return y_combined, loads

    def generate_deliveries(self, use_generated_paths:bool=True):
        """Generate delivery requests based on distribution."""
        global num_hotspots, full_time_matrix, full_dist_matrix, index_to_tract

        # All nodes in UMST are valid
        valid_nodes = list(range(num_hotspots))

        deliveries = []
        with tqdm(total=sum(self.loads)) as pbar:
            for j in range(60 * self.hours):
                for _ in range(self.loads[j]):
                    while True:
                        start = random.choice(valid_nodes)
                        end = random.choice(valid_nodes)

                        # Check path exists if required
                        if use_generated_paths and full_time_matrix[start, end] == sys.maxsize:
                            continue

                        # Must be different nodes
                        if start != end:
                            break

                    # Generate time within the minute
                    time = random.randint(j * 60, (j + 1) * 60)

                    # Create delivery
                    temp_delivery = Delivery(start, end, time, self.percent_max_dist)
                    deliveries.append(temp_delivery)
                    pbar.update(1)

        return deliveries

    @staticmethod
    def save_deliveries(deliverylist, graph:bool=False):
        """Save delivery list to file."""
        if not isinstance(deliverylist, DeliveryList):
            raise TypeError("Parameter must be DeliveryList type.")

        temp_dir = deliveries_dir
        if graph:
            temp_dir += "/Graphing"
            os.makedirs(temp_dir, exist_ok=True)

        filepath = f"{temp_dir}/{deliverylist.id}"
        if not graph:
            print(f"Saving to {filepath}")

        with open(filepath, 'wb') as file:
            pickle.dump(deliverylist, file)

    @staticmethod
    def load_deliveries(deliveries_id:str, graph:bool=False):
        """Load delivery list from file."""
        temp_dir = deliveries_dir
        if graph:
            temp_dir += "/Graphing"

        filepath = f"{temp_dir}/{deliveries_id}"
        if not graph:
            print(f"Loading from {filepath}")

        with open(filepath, 'rb') as file:
            deliveries = pickle.load(file)

        return deliveries

    def reset_deliveries(self, percent_max_dist:float=-1):
        """Reset all deliveries to initial state."""
        for d in self.deliveries:
            d.reset(percent_max_dist)

        self.percent_max_dist = percent_max_dist if percent_max_dist >= 0 else self.percent_max_dist
        self.id = DeliveryList.gen_deliveries_id(
            distance_based=self.distance_based,
            load=self.load,
            percent_max_dist=self.percent_max_dist,
            hours=self.hours,
            peaks=self.peaks,
            sigma=self.sigma,
            gen_multi=self.gen_multi
        )

    def copy(self):
        """Create deep copy of delivery list."""
        return copy.deepcopy(self)

    def __eq__(self, other):
        """Check equality of delivery lists."""
        if not isinstance(other, DeliveryList):
            return False

        return (self.distance_based == other.distance_based and
                self.load == other.load and
                self.hours == other.hours and
                self.peaks == other.peaks and
                self.sigma == other.sigma and
                self.percent_max_dist == other.percent_max_dist and
                np.array_equal(self.distribution, other.distribution))

    def __str__(self):
        return f"DeliveryList {self.id}\n" \
               f"Distance based: '{self.distance_based}'\n" \
               f"Load: {self.load}\n" \
               f"# Deliveries: {len(self.deliveries)}\n" \
               f"Hours: {self.hours}\n" \
               f"Peaks: {self.peaks}\n" \
               f"Sigma: {self.sigma}\n" \
               f"% Max Distance: {self.percent_max_dist}%\n"


### Simulation

In [ ]:
### Environment Class ####

class Environment:
    """
    Simulation environment for UMST-based delivery system.
    Supports bundling of multiple deliveries.
    """

    def __init__(self, deliverylist:DeliveryList, num_hotspots,
                 distance_matrix, time_matrix):
        global INTERVALS

        self.clock = 0
        self.time_step = 10
        self.num_hotspots = num_hotspots
        self.distance_matrix = distance_matrix
        self.time_matrix = time_matrix
        self.bundles_created = 0
        self.total_vehicles = 0
        self.vehicles = [0 for _ in range(self.num_hotspots)]

        self.deliverylist = deliverylist
        self.deliveries = deliverylist.deliveries
        self.active_deliveries = []
        self.node_deliveries = [[] for _ in range(self.num_hotspots)]
        self.dormant_deliveries = [d for d in self.deliveries]
        self.active_bundles = []

        # Metrics
        self.destination_success = np.zeros((num_hotspots,), dtype=int)
        self.source_success = np.zeros((num_hotspots,), dtype=int)
        self.order_rate = np.zeros((num_hotspots,), dtype=int)

    def run(self, grace_time:int, bundling=True, max_bundle_size=5, ):
        """
        Run simulation.

        Args:
            bundling: Enable delivery bundling
            max_bundle_size: Maximum deliveries per bundle
        """
        max_time = self.deliverylist.hours * 60 * 60 + grace_time

        while self.clock < max_time and self.active_deliveries or self.dormant_deliveries:
            self.step(bundling, max_bundle_size)

    def step(self, bundling=True, max_bundle_size=5):
        """Execute one simulation step."""
        # Add new deliveries
        self.add_deliveries()

        # Increment time
        self.clock += self.time_step

        # print(f"Steps-Clock: {self.clock}")

        # Assign deliveries to nodes
        self.assign_deliveries_to_nodes()

        # Create bundles if enabled
        if bundling:
            self.create_bundles(max_bundle_size)

        # Set next nodes for unbundled deliveries
        self.set_next_nodes()

        # Move deliveries forward
        self.move_deliveries()

        # Update vehicles
        self.update_vehicles()

    def add_deliveries(self):
        """Add deliveries whose start time has arrived."""
        for delivery in reversed(self.dormant_deliveries):
            if delivery.start_time <= self.clock:
                self.active_deliveries.append(delivery)
                self.dormant_deliveries.remove(delivery)

    def assign_deliveries_to_nodes(self):
        """Assign deliveries to their current nodes."""
        self.node_deliveries = [[] for _ in range(self.num_hotspots)]

        for delivery in self.active_deliveries:
            if not delivery.in_transition and not delivery.completed:
                self.node_deliveries[delivery.current_node].append(delivery)

    def create_bundles(self, max_bundle_size):
        """
        Create delivery bundles for efficient routing.

        Args:
            max_bundle_size: Maximum number of deliveries per bundle
        """
        # Find bundling opportunities at each node
        for node_idx in range(self.num_hotspots):
            node_dels = self.node_deliveries[node_idx]

            # Skip if too few deliveries or all bundled
            available_dels = [d for d in node_dels if d.bundle is None]
            if len(available_dels) < 2:
                continue

            # Group deliveries by compatibility
            bundles_to_create = self.find_bundle_groups(
                available_dels, max_bundle_size
            )

            # Create bundles
            for bundle_group in bundles_to_create:
                bundle = DeliveryBundle(
                    bundle_group,
                    self.distance_matrix,
                    self.time_matrix
                )
                self.active_bundles.append(bundle)
                self.bundles_created += 1

                # Mark bundle start node for all deliveries
                for d in bundle_group:
                    d.bundle_start_node = node_idx

    def find_bundle_groups(self, deliveries, max_size):
        """
        Find groups of deliveries that can be bundled together.
        Uses greedy clustering based on destination proximity.

        Args:
            deliveries: List of available deliveries
            max_size: Maximum bundle size

        Returns:
            list: List of delivery groups to bundle
        """
        if len(deliveries) < 2:
            return []

        bundles = []
        remaining = list(deliveries)

        while len(remaining) >= 2:
            # Start with first delivery
            seed = remaining[0]
            bundle_group = [seed]
            remaining.remove(seed)

            # Find compatible deliveries
            candidates = []
            for d in remaining:
                print(f"Bundling_feasibility: D1_st:{seed.start_time}:{seed.start_node}->{seed.end_node}, D2_st:{d.start_time}:{d.start_node}->{d.end_node}")
                if seed.check_bundling_feasibility(d, 300):
                    # Calculate compatibility score (lower is better)
                    dist = self.distance_matrix[seed.end_node, d.end_node]
                    candidates.append((dist, d))

            # Sort by distance and add to bundle
            candidates.sort(key=lambda x: x[0])
            for _, delivery in candidates[:max_size-1]:
                bundle_group.append(delivery)
                remaining.remove(delivery)

            print(f"Bundle Length: {len(bundle_group)}")
            # Only create bundle if multiple deliveries
            if len(bundle_group) >= 2:
                bundles.append(bundle_group)

        return bundles

    def set_next_nodes(self):
        """Set next nodes for unbundled deliveries."""
        for delivery in self.active_deliveries:
            if delivery.bundle is None and not delivery.in_transition and not delivery.completed:
                delivery.next_node = delivery.end_node
                delivery.path.append(delivery.next_node)
                self.order_rate[delivery.next_node] += 1
                delivery.distance_traveled += self.distance_matrix[
                    delivery.current_node, delivery.next_node
                ]
                delivery.time_till_next_node = self.time_matrix[
                    delivery.current_node, delivery.next_node
                ]

    def move_deliveries(self):
        """Move all deliveries forward in time."""
        to_remove = []
        vehicle_changes = [0 for _ in range(self.num_hotspots)]

        for delivery in self.active_deliveries:
            # Handle bundled deliveries
            if delivery.bundle is not None:
                self.move_bundled_delivery(delivery, vehicle_changes)
            else:
                # Move unbundled delivery
                if delivery.in_transition or delivery.next_node is not None:
                    delivery.time_till_next_node -= self.time_step

                    if not delivery.in_transition and delivery.is_primary:
                        vehicle_changes[delivery.current_node] -= 1

                    delivery.in_transition = True

                    if delivery.time_till_next_node <= 0:
                        delivery.in_transition = False
                        if delivery.is_primary:
                            self.vehicles[delivery.next_node] += 1

                        delivery.current_node = delivery.next_node

                        if delivery.current_node == delivery.end_node:
                            delivery.end_time = self.clock
                            delivery.completed = True
                            to_remove.append(delivery)

                        delivery.time_till_next_node = 0

        # Remove completed deliveries
        for delivery in to_remove:
            self.active_deliveries.remove(delivery)

        # Update vehicle counts
        for i in range(self.num_hotspots):
            self.vehicles[i] += vehicle_changes[i]
            if self.vehicles[i] < 0:
                self.total_vehicles += abs(self.vehicles[i])
                self.vehicles[i] = 0

    def move_bundled_delivery(self, delivery, vehicle_changes):
        """Move a delivery that's part of a bundle."""
        bundle = delivery.bundle

        # Get next node from bundle route
        if delivery.next_node is None:
            next_node = bundle.get_next_node()
            if next_node is not None:
                delivery.next_node = next_node
                delivery.path.append(next_node)
                self.order_rate[next_node] += 1

                if delivery.is_primary:
                    delivery.distance_traveled += self.distance_matrix[
                        delivery.current_node, next_node
                    ]

                delivery.time_till_next_node = self.time_matrix[
                    delivery.current_node, next_node
                ]

        # Move delivery
        if delivery.next_node is not None:
            delivery.time_till_next_node -= self.time_step

            if not delivery.in_transition and delivery.is_primary:
                vehicle_changes[delivery.current_node] -= 1

            delivery.in_transition = True

            if delivery.time_till_next_node <= 0:
                delivery.in_transition = False
                if delivery.is_primary:
                    self.vehicles[delivery.next_node] += 1

                delivery.current_node = delivery.next_node
                delivery.next_node = None

                # Check if reached destination
                if delivery.current_node == delivery.end_node:
                    delivery.end_time = self.clock
                    delivery.completed = True
                    bundle.remove_completed_deliveries()

                delivery.time_till_next_node = 0

                # Advance bundle route
                if delivery.is_primary and not delivery.completed:
                    bundle.advance_route()

    def update_vehicles(self):
        """Update vehicle availability at nodes."""
        pass  # Handled in move_deliveries

    def check_delivery_timelimit_success(self, delivery, time_val):
        """Check if delivery met time limit."""
        total_seconds = 3600 * self.deliverylist.hours
        sd = self.deliverylist.sigma / 60.0

        peak_diff = min(abs(total_seconds * peak - delivery.start_time)
                       for peak in self.deliverylist.peaks)
        num_sd_away = peak_diff / (total_seconds * sd)
        time_limit = delivery.time_limit * max(1, (2 - num_sd_away))

        return time_val < time_limit

    def results(self):
        """Calculate and return simulation results."""
        global data_dir, index_to_tract

        with open(data_dir + "/avg_hotspot_data.json") as f:
            hotspot_data = json.load(f)

        distances = []
        distances_tagged = []
        times = []
        times_tagged = []
        total_wait = 0
        total_hops = 0
        vehicle_change_times = self.calculate_vehicle_change_times()

        # Calculate success
        for delivery in self.deliveries:
            tract = index_to_tract[delivery.end_node]
            start_tract = index_to_tract[delivery.start_node]

            t = (delivery.end_time - delivery.start_time +
                 hotspot_data[tract]['time'] +
                 hotspot_data[start_tract]['time'] +
                 vehicle_change_times[delivery])

            if self.check_delivery_timelimit_success(delivery, t):
                delivery.successful = True
                self.destination_success[delivery.start_node] += 1
                self.source_success[delivery.end_node] += 1

        # Calculate metrics
        for delivery in self.deliveries:
            tract = index_to_tract[delivery.end_node]
            start_tract = index_to_tract[delivery.start_node]

            t = (delivery.end_time - delivery.start_time +
                 hotspot_data[tract]['time'] +
                 hotspot_data[start_tract]['time'] +
                 vehicle_change_times[delivery])

            d = (delivery.distance_traveled +
                 hotspot_data[start_tract]['distance'] +
                 hotspot_data[tract]['distance'])

            distances.append(d)
            distances_tagged.append((d, delivery.did_bundle, delivery.successful))
            times.append(t)
            times_tagged.append((t, delivery.did_bundle, delivery.successful))

            if delivery.successful:
                total_hops += len(delivery.path) - 1

                path_time = sum(self.time_matrix[delivery.path[i], delivery.path[i+1]]
                              for i in range(len(delivery.path)-1))

                total_wait += ((delivery.end_time - delivery.start_time) +
                             vehicle_change_times[delivery] - path_time)

        successful_deliveries = [d for d in self.deliveries if d.successful]
        successful_count = len(successful_deliveries)

        data = {
            'bundles': self.bundles_created,
            'success_rate': successful_count / len(self.deliveries),
            'success_rate_grouped': {},
            'description': "UMST-based delivery with bundling",
            'bundling': True,
            'distances': distances,
            'distances_tagged': distances_tagged,
            'times': times,
            'times_tagged': times_tagged,
            'wait': total_wait / max(1, successful_count),
            'hops': (total_hops + 2 * successful_count) / max(1, successful_count),
            'hops_all': (total_hops + 2 * len(self.deliveries)) / len(self.deliveries),
            'CDV': int(self.total_vehicles),
            'PDV': int(self.calculate_PDV()),
            'destination_success': self.destination_success,
            'source_success': self.source_success,
            'order_rate': self.order_rate,
            'avg_bundle_size': self.calculate_avg_bundle_size()
        }

        # Calculate success rate by time interval
        for interval in INTERVALS:
            interval_delivs = [d for d in self.deliveries if d.time_limit == interval]
            interval_success = [d for d in interval_delivs if d.successful]
            data['success_rate_grouped'][interval] = (
                len(interval_success) / max(1, len(interval_delivs))
            )

        return data

    def calculate_avg_bundle_size(self):
        """Calculate average bundle size."""
        bundled_deliveries = [d for d in self.deliveries if d.did_bundle]
        if not bundled_deliveries:
            return 0

        bundle_sizes = defaultdict(int)
        for d in bundled_deliveries:
            bundle_sizes[d.bundle_id] = len(d.bundled_with) + 1

        return sum(bundle_sizes.values()) / len(bundle_sizes) if bundle_sizes else 0

    def calculate_vehicle_change_times(self, fixed=True, factorMin=15, factorMax=30):
        """Calculate time spent changing vehicles for each delivery."""
        vehicle_change_times = {}

        for delivery in self.deliveries:
            if fixed:
                vehicle_change_times[delivery] = delivery.num_car_changes * factorMin
            else:
                tot = sum(random.randint(factorMin, factorMax)
                         for _ in range(delivery.num_car_changes))
                vehicle_change_times[delivery] = tot

        return vehicle_change_times

    def calculate_PDV(self):
        """Calculate Peak Delivery Vehicles (PDV) - maximum concurrent deliveries per node."""
        global data_dir, index_to_tract

        with open(data_dir + "/avg_hotspot_data.json") as f:
            hotspot_data = json.load(f)

        PDV_count = 0
        tolerance = 2 * self.time_step

        for node in range(self.num_hotspots):
            events = []

            for delivery in self.deliveries:
                tract = index_to_tract[node]

                if delivery.start_node == node:
                    start_time = delivery.start_time - hotspot_data[tract]["time"]
                    events.append((start_time + tolerance, 1))
                    events.append((start_time - tolerance, -1))

                if delivery.end_node == node and delivery.end_time is not None:
                    events.append((delivery.end_time + tolerance, 1))
                    events.append((delivery.end_time + hotspot_data[tract]["time"] - tolerance, -1))

            # Sort events and find maximum concurrent deliveries
            events.sort()

            max_active = 0
            active = 0
            for _, delta in events:
                active += delta
                max_active = max(max_active, active)

            PDV_count += max_active

        return PDV_count

    def baseline_no_bundling(self, grace_time:int):
        """
        Baseline: Direct delivery with no bundling on UMST.
        Each delivery travels directly from start to end.
        """
        self.reset()

        global data_dir, index_to_tract

        with open(data_dir + "/avg_hotspot_data.json") as f:
            hotspot_data = json.load(f)

        distances = []
        distances_tagged = []
        times = []
        times_tagged = []

        delivery_success_grouped = {interval: [0, 0] for interval in INTERVALS}

        successful_count = 0
        for delivery in self.deliveries:
            delivery_success_grouped[delivery.time_limit][1] += 1

            start_tract = index_to_tract[delivery.start_node]
            end_tract = index_to_tract[delivery.end_node]

            distance_val = (self.distance_matrix[delivery.start_node, delivery.end_node] +
                          hotspot_data[start_tract]["distance"] +
                          hotspot_data[end_tract]["distance"])

            time_val = (self.time_matrix[delivery.start_node, delivery.end_node] +
                       hotspot_data[start_tract]["time"] +
                       hotspot_data[end_tract]["time"])

            delivery_success = self.check_delivery_timelimit_success(delivery, time_val)

            if delivery_success:
                successful_count += 1
                delivery_success_grouped[delivery.time_limit][0] += 1

            distances.append(distance_val)
            times.append(time_val)
            distances_tagged.append((distance_val, False, delivery_success))
            times_tagged.append((time_val, False, delivery_success))

        # Run simulation without bundling for vehicle count
        self.reset()
        self.run(bundling=False, grace_time=grace_time)

        data = {
            'bundles': 0,
            'success_rate': successful_count / len(self.deliveries),
            'success_rate_grouped': {k: v[0] / max(1, v[1])
                                    for k, v in delivery_success_grouped.items()},
            'description': "UMST baseline - no bundling",
            'bundling': False,
            'distances': distances,
            'distances_tagged': distances_tagged,
            'times': times,
            'times_tagged': times_tagged,
            'wait': 0,
            'hops': 3,
            'hops_all': 3,
            'CDV': int(self.total_vehicles),
            'PDV': int(self.calculate_PDV()),
            'avg_bundle_size': 0
        }

        return data

    def reset(self):
        """Reset environment to initial state."""
        self.deliverylist.reset_deliveries()
        self.__init__(
            deliverylist=self.deliverylist,
            num_hotspots=self.num_hotspots,
            distance_matrix=self.distance_matrix,
            time_matrix=self.time_matrix
        )

    def __str__(self):
        return f"Environment\n" \
               f"Clock: {self.clock}\n" \
               f"Active Deliveries: {len(self.active_deliveries)}\n" \
               f"Dormant Deliveries: {len(self.dormant_deliveries)}\n" \
               f"Active Bundles: {len(self.active_bundles)}\n" \
               f"Bundles Created: {self.bundles_created}\n" \
               f"Total Vehicles: {self.total_vehicles}\n"




In [ ]:
#### Simulation Runner ####

def run_simulation(deliverylist, distance_matrix, time_matrix, num_hotspots, grace_time: int,
                   bundling=True, max_bundle_size=5):
    """
    Run a complete simulation.

    Args:
        deliverylist: DeliveryList object
        distance_matrix: Distance matrix from UMST
        time_matrix: Time matrix from UMST
        num_hotspots: Number of nodes in UMST
        bundling: Enable bundling
        max_bundle_size: Maximum deliveries per bundle

    Returns:
        dict: Simulation results
    """
    print("\nInitializing environment...")
    env = Environment(deliverylist, num_hotspots, distance_matrix, time_matrix)

    print("Running simulation...")
    env.run(bundling=bundling, max_bundle_size=max_bundle_size, grace_time=grace_time)

    print("Calculating results...")
    results = env.results()

    print("\n" + "="*50)
    print("SIMULATION RESULTS")
    print("="*50)
    print(f"Success Rate: {results['success_rate']:.2%}")
    print(f"Bundles Created: {results['bundles']}")
    print(f"Avg Bundle Size: {results['avg_bundle_size']:.2f}")
    print(f"Avg Hops (Successful): {results['hops']:.2f}")
    print(f"Avg Wait Time: {results['wait']:.2f} seconds")
    print(f"CDV (Created Vehicles): {results['CDV']}")
    print(f"PDV (Peak Vehicles): {results['PDV']}")
    # print(f"Total Distance: {sum(results['distances'])/1000:.2f} km")
    print(f"Avg Distance: {np.mean(results['distances'])/1000:.2f} km")
    print("="*50)

    return results


def compare_with_baseline(deliverylist, distance_matrix, time_matrix, num_hotspots, grace_time,
                         max_bundle_size=5):
    """
    Compare bundling approach with baseline (no bundling).

    Args:
        deliverylist: DeliveryList object
        distance_matrix: Distance matrix from UMST
        time_matrix: Time matrix from UMST
        num_hotspots: Number of nodes in UMST
        max_bundle_size: Maximum deliveries per bundle

    Returns:
        tuple: (bundling_results, baseline_results)
    """
    print("\n" + "="*50)
    print("RUNNING BUNDLING SIMULATION")
    print("="*50)
    bundling_results = run_simulation(
        deliverylist, distance_matrix, time_matrix, num_hotspots,
        bundling=True, max_bundle_size=max_bundle_size, grace_time=grace_time
    )

    print("\n" + "="*50)
    print("RUNNING BASELINE SIMULATION")
    print("="*50)
    env = Environment(deliverylist, num_hotspots, distance_matrix, time_matrix)
    baseline_results = env.baseline_no_bundling(grace_time=grace_time)

    print("\n" + "="*50)
    print("COMPARISON SUMMARY")
    print("="*50)
    print(f"{'Metric':<30} {'Bundling':<15} {'Baseline':<15} {'Improvement':<15}")
    print("-"*75)

    metrics = [
        ('Success Rate', 'success_rate', lambda x: f"{x:.2%}", lambda b, a: f"{((b-a)/a)*100:+.1f}%"),
        ('Avg Distance (km)', 'distances', lambda x: f"{np.mean(x)/1000:.2f}",
         lambda b, a: f"{((np.mean(b)-np.mean(a))/np.mean(a))*100:+.1f}%"),
        ('Avg Time (sec)', 'times', lambda x: f"{np.mean(x):.0f}",
         lambda b, a: f"{((np.mean(b)-np.mean(a))/np.mean(a))*100:+.1f}%"),
        ('Avg Hops', 'hops', lambda x: f"{x:.2f}",
         lambda b, a: f"{((b-a)/a)*100:+.1f}%"),
        ('CDV', 'CDV', lambda x: f"{x}", lambda b, a: f"{((b-a)/a)*100:+.1f}%"),
        ('PDV', 'PDV', lambda x: f"{x}", lambda b, a: f"{((b-a)/a)*100:+.1f}%"),
    ]

    for name, key, fmt, imp in metrics:
        bundling_val = bundling_results[key]
        baseline_val = baseline_results[key]
        print(f"{name:<30} {fmt(bundling_val):<15} {fmt(baseline_val):<15} {imp(bundling_val, baseline_val):<15}")

    print(f"{'Bundles Created':<30} {bundling_results['bundles']:<15} {'-':<15} {'-':<15}")
    print(f"{'Avg Bundle Size':<30} {bundling_results['avg_bundle_size']:.2f}{'':>12} {'-':<15} {'-':<15}")
    print("="*75)

    return bundling_results, baseline_results


## Final Function Call

In [ ]:
# ============================================================
# Define simulation parameters
# ============================================================

distance_based = "one_cluster"      # or "further", "one_cluster"
load = 100
percent_max_dist = 90
hours = 1
grace_time = 36000
peaks = [0.25, 0.75]
sigma = 10
gen_multi = False
use_generated_paths = True

bundling = True
max_bundle_size = 5

In [ ]:
# ============================================================
# 1. Generate or load delivery list
# ============================================================

deliverylist = DeliveryList(
    distance_based=distance_based,
    load=load,
    percent_max_dist=percent_max_dist,
    hours=hours,
    peaks=peaks,
    sigma=sigma,
    gen_multi=gen_multi,
    use_generated_paths=use_generated_paths
)

print(f"Total deliveries: {len(deliverylist.deliveries)}")

# Or load existing:
# deliverylist = DeliveryList.load_deliveries("your-delivery-id")


# ============================================================
# 2. Run simulation with bundling
# ============================================================
# results = run_simulation(
#     deliverylist=deliverylist,
#     distance_matrix=full_dist_matrix,
#     time_matrix=full_time_matrix,
#     num_hotspots=num_hotspots,
#     bundling=bundling,
#     max_bundle_size=max_bundle_size,
#     grace_time=grace_time
# )

# ============================================================
# 3. Compare with baseline
# ============================================================

results, baseline_results = compare_with_baseline(
    deliverylist=deliverylist,
    distance_matrix=full_dist_matrix,
    time_matrix=full_time_matrix,
    num_hotspots=num_hotspots,
    max_bundle_size=max_bundle_size,
    grace_time=grace_time
)

# ============================================================
# 4. Access specific metrics
# ============================================================

print(f"Success rate: {results['success_rate']}")
print(f"Total bundles: {results['bundles']}")
# print(f"Total distance: {sum(results['distances'])} meters")


Generating ID...
done.

Generating distribution...
done.

Generating deliveries...


100%|██████████| 4604/4604 [00:00<00:00, 14445.75it/s]



done.

Saving deliveries...
Saving to ./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data/Deliveries/3-0100-001-010-[0.25, 0.75]-090
done.
Total deliveries: 4604

RUNNING BUNDLING SIMULATION

Initializing environment...
Running simulation...
Bundling_feasibility: D1_st:48:2->18, D2_st:42:2->13
Return True - bundling feasible
Bundle Length: 2
Bundling_feasibility: D1_st:54:25->5, D2_st:56:25->5
Return True - same destination (PERFECT bundling, zero detour!)
Bundle Length: 2
Bundling_feasibility: D1_st:69:12->10, D2_st:65:12->16
Return True - bundling feasible
Bundling_feasibility: D1_st:69:12->10, D2_st:70:12->22
Return True - bundling feasible
Bundle Length: 3
Bundling_feasibility: D1_st:66:13->25, D2_st:62:13->11
Return True - bundling feasible
Bundle Length: 2
Bundling_feasibility: D1_st:77:1->24, D2_st:76:1->14
Return True - bundling feasible
Bundle Length: 2
Bundling_feasibility: D1_st:78:5->13, D2_st:75:5->11
Return True - bundling feasible
Bundle Lengt

In [ ]:
errrorrrrr

NameError: name 'errrorrrrr' is not defined

## DeliverList Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import viridis
from matplotlib.colors import Normalize

# Extract delivery information
deliveries = deliverylist.deliveries

print(f"Total deliveries: {len(deliveries)}")
print(f"Number of hotspots (census tracts): {num_hotspots}")

# Create index mapping dictionaries
index_to_tract = {}
tract_to_index = {}

for idx, row in census_gdf.iterrows():
    geoid = row['GEOID']
    index_to_tract[idx] = geoid
    tract_to_index[geoid] = idx

print(f"Created mapping for {len(index_to_tract)} census tracts")

# Extract delivery details
delivery_data = []
for i, delivery in enumerate(deliveries):
    data = {
        'delivery_id': i,
        'start_node': getattr(delivery, 'start_node', None),
        'end_node': getattr(delivery, 'end_node', None),
        'start_time': getattr(delivery, 'start_time', None),  # This is in seconds
        'num_shared': getattr(delivery, 'num_shared', 0) if hasattr(delivery, 'num_shared') else (len(delivery.shared) if hasattr(delivery, 'shared') and delivery.shared else 0)
    }
    delivery_data.append(data)

# Get hotspot locations using index mapping
index_to_location = {}
geoid_to_location = {}

for idx, row in census_gdf.iterrows():
    geoid = row['GEOID']
    centroid = row.geometry.centroid
    location = (centroid.x, centroid.y)

    index_to_location[idx] = location
    geoid_to_location[geoid] = location

print(f"Location mappings created: {len(index_to_location)}")

# Determine which mapping to use
def get_location(node):
    """Get location for a node, handling both index and GEOID formats"""
    if node is None:
        return None

    # Try as string (GEOID)
    node_str = str(node)
    if node_str in geoid_to_location:
        return geoid_to_location[node_str]

    # Try as integer index
    try:
        node_int = int(float(node_str))
        if node_int in index_to_location:
            return index_to_location[node_int]
        # Try mapping index to GEOID first
        if node_int in index_to_tract:
            geoid = index_to_tract[node_int]
            if geoid in geoid_to_location:
                return geoid_to_location[geoid]
    except:
        pass

    return None

# Select sample deliveries to show
num_samples = min(50, len(delivery_data))
sample_deliveries = delivery_data[:num_samples]

# === PREPARE TIME-BASED DATA (IN SECONDS) ===
# Start times are already in seconds
start_times_seconds = [d['start_time'] for d in delivery_data if d['start_time'] is not None]
start_times_minutes = [t / 60.0 for t in start_times_seconds]  # Convert seconds to minutes for display

# Simulation duration
simulation_duration_seconds = deliverylist.hours * 3600  # hours to seconds
simulation_duration_minutes = deliverylist.hours * 60  # hours to minutes

# Create bins for each minute (0-60 minutes = 0-3600 seconds)
num_minutes = int(simulation_duration_minutes)
minute_bins_seconds = np.arange(0, simulation_duration_seconds + 60, 60)  # Bins in seconds (0, 60, 120, ..., 3600)
delivery_counts, bin_edges = np.histogram(start_times_seconds, bins=minute_bins_seconds)

print(f"\nTime Analysis:")
print(f"  Simulation duration: {deliverylist.hours} hour(s) = {simulation_duration_minutes} minutes = {simulation_duration_seconds} seconds")
print(f"  Deliveries with time data: {len(start_times_seconds)}")
if start_times_seconds:
    print(f"  Time range: {min(start_times_seconds):.2f} - {max(start_times_seconds):.2f} seconds")
    print(f"              ({min(start_times_seconds)/60:.2f} - {max(start_times_seconds)/60:.2f} minutes)")

# Create visualization with 3 plots
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 2, height_ratios=[2, 2, 1.5])

fig.suptitle(f'Delivery Analysis (Load: {deliverylist.load}, Total: {len(deliveries)} deliveries, Duration: {num_minutes} minutes)',
             fontsize=16, fontweight='bold')

# === PLOT 1: Deliveries with Start/End Nodes Highlighted ===
ax1 = fig.add_subplot(gs[0, 0])

# Plot census tracts
if census_gdf is not None:
    census_gdf.boundary.plot(ax=ax1, color='gray', linewidth=0.8, alpha=0.6)
    census_gdf.plot(ax=ax1, color='lightgray', alpha=0.15, edgecolor='gray', linewidth=0.5)

# Plot all hotspot centroids (faded)
for idx, loc in index_to_location.items():
    ax1.scatter(loc[0], loc[1], s=30, c='lightgray', marker='o',
               edgecolors='gray', linewidths=0.5, alpha=0.3, zorder=2)

# Get start times for coloring (in seconds)
start_times_sample = [d['start_time'] for d in sample_deliveries if d['start_time'] is not None]

if start_times_sample:
    norm = Normalize(vmin=min(start_times_sample), vmax=max(start_times_sample))
    cmap = viridis

    # Plot each delivery route with colored line
    routes_plotted = 0
    for data in sample_deliveries:
        if data['start_node'] is not None and data['end_node'] is not None and data['start_time'] is not None:
            start_loc = get_location(data['start_node'])
            end_loc = get_location(data['end_node'])

            if start_loc and end_loc:
                color = cmap(norm(data['start_time']))

                # Draw route line
                ax1.plot([start_loc[0], end_loc[0]], [start_loc[1], end_loc[1]],
                        color=color, alpha=0.7, linewidth=2.5, zorder=4)

                # Mark start node (circle)
                ax1.scatter(start_loc[0], start_loc[1], s=180, c=[color],
                           marker='o', edgecolors='black', linewidths=2.5,
                           alpha=0.9, zorder=5)

                # Mark end node (square)
                ax1.scatter(end_loc[0], end_loc[1], s=180, c=[color],
                           marker='s', edgecolors='black', linewidths=2.5,
                           alpha=0.9, zorder=5)

                routes_plotted += 1

    print(f"Routes plotted in sample: {routes_plotted}")

    # Add colorbar (show in minutes for readability)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar1 = plt.colorbar(sm, ax=ax1, label='Start Time (seconds)', fraction=0.046, pad=0.04)

# Add legend for markers
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue',
           markeredgecolor='black', markersize=10, label='Start Node'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='blue',
           markeredgecolor='black', markersize=10, label='End Node'),
    Line2D([0], [0], color='purple', linewidth=2, label='Delivery Route')
]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=10)

ax1.set_xlabel('Longitude', fontsize=12)
ax1.set_ylabel('Latitude', fontsize=12)
ax1.set_title(f'Sample Delivery Routes (First {num_samples}) with Start (○) and End (□) Nodes', fontsize=13)
ax1.set_aspect('equal', adjustable='box')

# === PLOT 2: All Deliveries Overview with Density ===
ax2 = fig.add_subplot(gs[0, 1])

# Plot census tracts
if census_gdf is not None:
    census_gdf.boundary.plot(ax=ax2, color='gray', linewidth=0.8, alpha=0.6)
    census_gdf.plot(ax=ax2, color='lightgray', alpha=0.15, edgecolor='gray', linewidth=0.5)

# Count deliveries per hotspot for all deliveries
hotspot_activity = {}
for data in delivery_data:
    if data['start_node'] is not None:
        hotspot_activity[data['start_node']] = hotspot_activity.get(data['start_node'], 0) + 1
    if data['end_node'] is not None:
        hotspot_activity[data['end_node']] = hotspot_activity.get(data['end_node'], 0) + 1

# Plot hotspots sized and colored by activity
activities = []
locs_x = []
locs_y = []

for node, count in hotspot_activity.items():
    loc = get_location(node)
    if loc:
        activities.append(count)
        locs_x.append(loc[0])
        locs_y.append(loc[1])

if activities:
    # Plot with size proportional to activity
    sizes = [50 + a * 15 for a in activities]
    scatter = ax2.scatter(locs_x, locs_y, s=sizes, c=activities,
                         cmap='YlOrRd', edgecolors='black', linewidths=1,
                         alpha=0.7, zorder=5)

    # Add colorbar
    cbar2 = plt.colorbar(scatter, ax=ax2, label='Number of Deliveries',
                        fraction=0.046, pad=0.04)

# Plot all delivery routes (faded)
routes_total = 0
for data in delivery_data:
    if data['start_node'] is not None and data['end_node'] is not None:
        start_loc = get_location(data['start_node'])
        end_loc = get_location(data['end_node'])
        if start_loc and end_loc:
            ax2.plot([start_loc[0], end_loc[0]], [start_loc[1], end_loc[1]],
                    'b-', alpha=0.05, linewidth=0.5, zorder=2)
            routes_total += 1

print(f"Total routes plotted: {routes_total}")

ax2.set_xlabel('Longitude', fontsize=12)
ax2.set_ylabel('Latitude', fontsize=12)
ax2.set_title(f'All {len(deliveries)} Deliveries - Hotspot Activity', fontsize=13)
ax2.set_aspect('equal', adjustable='box')

# === PLOT 3: Bar Chart - Deliveries per Minute ===
ax3 = fig.add_subplot(gs[1, :])

# Create bar chart (x-axis in minutes, but data binned from seconds)
minutes = np.arange(0, num_minutes)
bars = ax3.bar(minutes, delivery_counts, color='steelblue', edgecolor='black',
               linewidth=0.5, alpha=0.8)

# Highlight peak periods (peaks are in hours, convert to minutes)
# peaks=[0.25, 0.75] means 0.25*60=15 min and 0.75*60=45 min
peak_minutes = [p * 60 for p in deliverylist.peaks]
for peak_min in peak_minutes:
    if 0 <= peak_min < num_minutes:
        ax3.axvline(peak_min, color='red', linestyle='--', linewidth=2,
                   alpha=0.7, label=f'Peak at {peak_min:.0f} min' if peak_min == peak_minutes[0] else '')

# Add statistics
if len(delivery_counts) > 0:
    max_count = max(delivery_counts)
    max_minute = minutes[np.argmax(delivery_counts)]
    avg_count = np.mean(delivery_counts)

    ax3.axhline(avg_count, color='green', linestyle=':', linewidth=2,
               alpha=0.7, label=f'Average: {avg_count:.1f} deliveries/min')

    # Annotations
    ax3.text(max_minute, max_count, f'Max: {int(max_count)}\n@{int(max_minute)}min',
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax3.set_xlabel('Time (minutes)', fontsize=12)
ax3.set_ylabel('Number of Deliveries Starting', fontsize=12)
ax3.set_title(f'Delivery Distribution Over {num_minutes}-Minute Simulation', fontsize=13, fontweight='bold')
ax3.set_xlim(-1, num_minutes)
ax3.set_xticks(np.arange(0, num_minutes + 1, 5))
ax3.grid(True, alpha=0.3, axis='y')
ax3.legend(loc='upper right', fontsize=10)

# === PLOT 4: Statistics Summary ===
ax4 = fig.add_subplot(gs[2, :])

# Calculate time-based statistics
if start_times_seconds and len(delivery_counts) > 0:
    time_stats = f"""
TEMPORAL DISTRIBUTION STATISTICS
{'='*80}

Total Deliveries: {len(deliveries)}
Deliveries with Time Data: {len(start_times_seconds)}

TIME DISTRIBUTION:
  Simulation Duration: {simulation_duration_minutes:.0f} minutes ({simulation_duration_seconds:.0f} seconds)
  Peak Times (configured): {', '.join([f'{p*60:.0f} min ({p*3600:.0f}s)' for p in deliverylist.peaks])}
  Actual Peak Time: Minute {int(max_minute)} with {int(max_count)} deliveries

  Min deliveries/minute: {int(min(delivery_counts))}
  Max deliveries/minute: {int(max_count)}
  Average deliveries/minute: {avg_count:.2f}
  Standard deviation: {np.std(delivery_counts):.2f}

DISTRIBUTION CHARACTERISTICS:
  First delivery at: {min(start_times_seconds):.2f} seconds ({min(start_times_seconds)/60:.2f} minutes)
  Last delivery at: {max(start_times_seconds):.2f} seconds ({max(start_times_seconds)/60:.2f} minutes)
  Median delivery time: {np.median(start_times_seconds):.2f} seconds ({np.median(start_times_seconds)/60:.2f} minutes)

LOAD PARAMETER EXPLANATION:
  Load = {deliverylist.load} acts as a scaling factor
  This generated {len(deliveries)} deliveries ({len(deliveries) / deliverylist.load:.1f} per load unit)
  Distributed across {num_hotspots} available hotspots
  Using {len(set([d['start_node'] for d in delivery_data if d['start_node']] + [d['end_node'] for d in delivery_data if d['end_node']]))} unique hotspots ({len(set([d['start_node'] for d in delivery_data if d['start_node']] + [d['end_node'] for d in delivery_data if d['end_node']]))/num_hotspots*100:.1f}% utilization)
"""
else:
    time_stats = "No time data available"

ax4.text(0.05, 0.5, time_stats, fontsize=10, family='monospace',
        verticalalignment='center', transform=ax4.transAxes)
ax4.axis('off')

plt.tight_layout()
plt.show()

# Print minute-by-minute breakdown for first 20 minutes
print("\n" + "="*60)
print("DELIVERIES PER MINUTE (First 20 minutes):")
print("="*60)
print(f"{'Minute':<10} {'Time Range (s)':<20} {'Count':<10} {'Bar':<30}")
print("-"*60)
for i in range(min(20, len(delivery_counts))):
    time_range = f"{i*60}-{(i+1)*60}"
    bar = '█' * int(delivery_counts[i])
    print(f"{i:<10} {time_range:<20} {int(delivery_counts[i]):<10} {bar}")

print(f"\n... Total across all {num_minutes} minutes: {int(sum(delivery_counts))} deliveries")

# Print sample deliveries with time in seconds
print("\n" + "="*100)
print(f"SAMPLE DELIVERIES (First 10):")
print("="*100)
print(f"{'ID':<5} {'Start Node':<12} {'End Node':<12} {'Start Time (s)':<15} {'Start Time (min)':<15} {'Shared':<8}")
print("-"*100)

for i, data in enumerate(delivery_data[:10]):
    start_node = str(data['start_node']) if data['start_node'] is not None else "N/A"
    end_node = str(data['end_node']) if data['end_node'] is not None else "N/A"
    start_time_s = f"{data['start_time']:.2f}" if data['start_time'] is not None else "N/A"
    start_time_m = f"{data['start_time']/60:.2f}" if data['start_time'] is not None else "N/A"

    print(f"{i+1:<5} {start_node:<12} {end_node:<12} {start_time_s:<15} {start_time_m:<15} {data['num_shared']:<8}")

## Debug

In [ ]:
import time
import numpy as np
from collections import defaultdict


class SimulationDebugger:
    def __init__(self):
        self.step_times = []
        self.operation_times = defaultdict(list)
        self.step_counts = defaultdict(int)

    def time_operation(self, name):
        class Timer:
            def __init__(self, debugger, op_name):
                self.debugger = debugger
                self.op_name = op_name
                self.start = None

            def __enter__(self):
                self.start = time.time()
                return self

            def __exit__(self, *args):
                elapsed = time.time() - self.start
                self.debugger.operation_times[self.op_name].append(elapsed)

        return Timer(self, name)

    def print_report(self):
        print("\n" + "="*70)
        print("SIMULATION PERFORMANCE REPORT")
        print("="*70)

        for op_name, times in sorted(self.operation_times.items()):
            total = sum(times)
            avg = total / len(times) if times else 0
            calls = len(times)
            print(f"{op_name:<30} {total:>8.2f}s  {avg*1000:>8.2f}ms/call  {calls:>6} calls")

        print("="*70)


def debug_environment_step(env, max_steps=100, verbose=True):
    debugger = SimulationDebugger()
    step = 0

    print(f"Starting debug simulation (max {max_steps} steps)...")
    print(f"Initial state: {len(env.dormant_deliveries)} dormant, {len(env.active_deliveries)} active")

    while (env.active_deliveries or env.dormant_deliveries) and step < max_steps:
        step_start = time.time()

        with debugger.time_operation("add_deliveries"):
            env.add_deliveries()

        env.clock += env.time_step

        with debugger.time_operation("assign_to_nodes"):
            env.assign_deliveries_to_nodes()

        with debugger.time_operation("create_bundles"):
            env.create_bundles(max_bundle_size=5)

        with debugger.time_operation("set_next_nodes"):
            env.set_next_nodes()

        with debugger.time_operation("move_deliveries"):
            env.move_deliveries()

        with debugger.time_operation("update_vehicles"):
            env.update_vehicles()

        step_time = time.time() - step_start
        debugger.step_times.append(step_time)

        if verbose and step % 10 == 0:
            print(f"Step {step:4d} | Clock: {env.clock:6d}s | "
                  f"Active: {len(env.active_deliveries):4d} | "
                  f"Dormant: {len(env.dormant_deliveries):4d} | "
                  f"Bundles: {len(env.active_bundles):3d} | "
                  f"Time: {step_time*1000:.1f}ms")

        step += 1

    if step >= max_steps:
        print(f"\n⚠ Stopped at max steps ({max_steps})")
    else:
        print(f"\n✓ Simulation completed in {step} steps")

    debugger.print_report()

    print(f"\nStep statistics:")
    print(f"  Total steps: {len(debugger.step_times)}")
    print(f"  Avg step time: {np.mean(debugger.step_times)*1000:.2f}ms")
    print(f"  Max step time: {np.max(debugger.step_times)*1000:.2f}ms")
    print(f"  Total time: {sum(debugger.step_times):.2f}s")

    return debugger


def profile_bundling(env, max_checks=1000):
    print("\nProfiling bundling performance...")

    # Simulate some deliveries at nodes
    test_deliveries = env.deliveries[:min(100, len(env.deliveries))]

    # Group by node
    node_groups = defaultdict(list)
    for d in test_deliveries:
        if not d.completed and d.bundle is None:
            node_groups[d.start_node].append(d)

    bundle_times = []
    total_comparisons = 0

    for node, deliveries in list(node_groups.items())[:10]:
        if len(deliveries) < 2:
            continue

        start = time.time()

        # Test bundling logic
        for i, d1 in enumerate(deliveries):
            for d2 in deliveries[i+1:]:
                total_comparisons += 1
                d1.check_bundling_feasibility(d2)

                if total_comparisons >= max_checks:
                    break
            if total_comparisons >= max_checks:
                break

        elapsed = time.time() - start
        bundle_times.append(elapsed)

        if total_comparisons >= max_checks:
            break

    if bundle_times:
        avg_time = np.mean(bundle_times)
        print(f"  Comparisons: {total_comparisons}")
        print(f"  Avg time per node: {avg_time*1000:.2f}ms")
        print(f"  Time per comparison: {sum(bundle_times)/total_comparisons*1000000:.2f}µs")


def check_matrix_performance(distance_matrix, time_matrix, num_samples=1000):
    print("\nChecking matrix access performance...")

    n = len(distance_matrix)

    # Random accesses
    start = time.time()
    for _ in range(num_samples):
        i, j = np.random.randint(0, n, 2)
        _ = distance_matrix[i, j]
        _ = time_matrix[i, j]
    elapsed = time.time() - start

    print(f"  {num_samples} random accesses: {elapsed*1000:.2f}ms")
    print(f"  Avg per access: {elapsed/num_samples*1000000:.2f}µs")

    # Check for inf values
    inf_count = np.sum(np.isinf(distance_matrix))
    print(f"  Inf values in distance matrix: {inf_count}")

    if inf_count > n:
        print(f"  ⚠ WARNING: Graph may not be fully connected!")


def analyze_delivery_distribution(deliverylist):
    print("\nDelivery Distribution Analysis:")
    print(f"  Total deliveries: {len(deliverylist.deliveries)}")

    # Time distribution
    start_times = [d.start_time for d in deliverylist.deliveries]
    print(f"  Start time range: {min(start_times)} - {max(start_times)}s")
    print(f"  Avg start time: {np.mean(start_times):.0f}s")

    # Distance distribution
    distances = []
    for d in deliverylist.deliveries:
        dist = full_dist_matrix[d.start_node, d.end_node]
        if not np.isinf(dist):
            distances.append(dist)

    if distances:
        print(f"  Valid paths: {len(distances)}/{len(deliverylist.deliveries)}")
        print(f"  Avg distance: {np.mean(distances):.3f} km")
        print(f"  Max distance: {np.max(distances):.3f} km")
    else:
        print(f"  ⚠ WARNING: No valid paths found!")


def quick_test_simulation(deliverylist, distance_matrix, time_matrix, num_hotspots):
    print("\n" + "="*70)
    print("QUICK TEST SIMULATION (10 deliveries, 100 steps max)")
    print("="*70)

    # Create small test
    test_list = deliverylist.copy()
    test_list.deliveries = deliverylist.deliveries[:10]

    env = Environment(test_list, num_hotspots, distance_matrix, time_matrix)

    debugger = debug_environment_step(env, max_steps=100, verbose=True)

    return debugger


def full_debug_suite(deliverylist, distance_matrix, time_matrix, num_hotspots):
    print("\n" + "="*70)
    print("FULL DEBUGGING SUITE")
    print("="*70)

    # 1. Matrix check
    check_matrix_performance(distance_matrix, time_matrix)

    # 2. Delivery analysis
    analyze_delivery_distribution(deliverylist)

    # 3. Quick simulation test
    quick_test_simulation(deliverylist, distance_matrix, time_matrix, num_hotspots)

    print("\n" + "="*70)
    print("DEBUGGING COMPLETE")
    print("="*70)


In [ ]:
# Usage examples:

# 1. Debug a full simulation with performance tracking

# env = Environment(deliverylist, num_hotspots, full_dist_matrix, full_time_matrix)
# debugger = debug_environment_step(env, max_steps=500, verbose=True)


# 2. Run full debug suite

# full_debug_suite(deliverylist, full_dist_matrix, full_time_matrix, num_hotspots)


# 3. Profile just bundling

env = Environment(deliverylist, num_hotspots, full_dist_matrix, full_time_matrix)
profile_bundling(env)


# 4. Check matrices only

check_matrix_performance(full_dist_matrix, full_time_matrix)


# Fine Tunining the Model

In [ ]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy.spatial.distance import cdist
import os

def run_parameter_sweep(
    # Fixed parameters
    distance_matrix,
    time_matrix,
    num_hotspots,
    census_gdf,

    # Parameter ranges to test
    distance_based_options=["closer", "further", "one_cluster"],
    load_options=[50, 100, 150, 200],
    percent_max_dist_options=[70, 80, 90, 100],
    bundling_options=[True, False],
    max_bundle_size_options=[2, 3, 4, 5],
    hours_options=[1],
    peaks_options=[[0.25, 0.75]],
    sigma_options=[10],
    grace_time_options=[1800],  # 30 minutes

    # Other settings
    num_trials=3,  # Run multiple times for each config to average results
    save_results=True,
    output_dir="./experiment_results/"
):
    """
    Run comprehensive parameter sweep for delivery simulation.

    Returns:
        DataFrame with results for all parameter combinations
    """

    # Create output directory
    if save_results:
        os.makedirs(output_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Generate all parameter combinations
    param_combinations = list(itertools.product(
        distance_based_options,
        load_options,
        percent_max_dist_options,
        bundling_options,
        max_bundle_size_options,
        hours_options,
        peaks_options,
        sigma_options,
        grace_time_options
    ))

    print(f"Total combinations to test: {len(param_combinations)}")
    print(f"Trials per combination: {num_trials}")
    print(f"Total simulations: {len(param_combinations) * num_trials}")
    print("="*80)

    results_list = []

    for idx, (distance_based, load, percent_max_dist, bundling, max_bundle_size,
              hours, peaks, sigma, grace_time) in enumerate(param_combinations, 1):

        # Skip invalid combinations
        if not bundling and max_bundle_size > 1:
            if max_bundle_size != max_bundle_size_options[0]:
                continue

        print(f"\n[{idx}/{len(param_combinations)}] Testing configuration:")
        print(f"  distance_based={distance_based}, load={load}, "
              f"percent_max_dist={percent_max_dist}, bundling={bundling}, "
              f"max_bundle_size={max_bundle_size}")

        # Run multiple trials for this configuration
        trial_results = []

        for trial in range(1, num_trials + 1):
            print(f"  Trial {trial}/{num_trials}...", end=" ")

            try:
                # Generate delivery list
                deliverylist = DeliveryList(
                    distance_based=distance_based,
                    load=load,
                    percent_max_dist=percent_max_dist,
                    hours=hours,
                    peaks=peaks,
                    sigma=sigma,
                    gen_multi=True,
                    multi_id=trial,
                    use_generated_paths=True
                )

                # Run simulation
                sim_results = run_simulation(
                    deliverylist=deliverylist,
                    distance_matrix=distance_matrix,
                    time_matrix=time_matrix,
                    num_hotspots=num_hotspots,
                    bundling=bundling,
                    max_bundle_size=max_bundle_size if bundling else 1,
                    grace_time=grace_time
                )

                # Calculate percentile metrics
                delivery_times = sim_results.get('delivery_times', [])
                delivery_distances = sim_results.get('delivery_distances', [])

                # Store results with comprehensive metrics
                result = {
                    # Parameters
                    'distance_based': distance_based,
                    'load': load,
                    'percent_max_dist': percent_max_dist,
                    'bundling': bundling,
                    'max_bundle_size': max_bundle_size if bundling else 1,
                    'hours': hours,
                    'peaks': str(peaks),
                    'sigma': sigma,
                    'grace_time': grace_time,
                    'trial': trial,

                    # Primary Metrics (for Pareto analysis)
                    'total_deliveries': len(deliverylist.deliveries),
                    'avg_distance': sim_results['avg_distance'],
                    'avg_time': sim_results['avg_delivery_time'],
                    'total_vehicles': sim_results.get('total_vehicles', sim_results['bundles']),

                    # Success metrics
                    'success_rate': sim_results['success_rate'],
                    'completion_rate': sim_results['completion_rate'],

                    # Detailed distance metrics
                    'total_distance': sim_results['total_distance'],
                    'median_distance': np.median(delivery_distances) if delivery_distances else 0,
                    'p90_distance': np.percentile(delivery_distances, 90) if delivery_distances else 0,
                    'p95_distance': np.percentile(delivery_distances, 95) if delivery_distances else 0,
                    'max_distance': np.max(delivery_distances) if delivery_distances else 0,
                    'std_distance': np.std(delivery_distances) if delivery_distances else 0,

                    # Detailed time metrics
                    'median_time': np.median(delivery_times) if delivery_times else 0,
                    'p90_time': np.percentile(delivery_times, 90) if delivery_times else 0,
                    'p95_time': np.percentile(delivery_times, 95) if delivery_times else 0,
                    'p99_time': np.percentile(delivery_times, 99) if delivery_times else 0,
                    'max_time': np.max(delivery_times) if delivery_times else 0,
                    'std_time': np.std(delivery_times) if delivery_times else 0,

                    # Bundling metrics
                    'num_bundles': sim_results['bundles'],
                    'avg_bundle_size': sim_results['avg_bundle_size'],
                    'bundling_efficiency': sim_results.get('bundling_efficiency', 0),

                    # Temporal metrics
                    'simulation_time': sim_results['simulation_time'],
                    'deliveries_in_window': sim_results['deliveries_in_window'],
                    'deliveries_in_grace': sim_results['deliveries_in_grace'],
                    'incomplete_deliveries': sim_results['incomplete_deliveries'],

                    # Efficiency metrics
                    'distance_per_delivery': sim_results['avg_distance'],
                    'time_per_delivery': sim_results['avg_delivery_time'],
                    'deliveries_per_vehicle': len(deliverylist.deliveries) / max(sim_results.get('total_vehicles', 1), 1),
                }

                trial_results.append(result)
                print("✓")

            except Exception as e:
                print(f"✗ Error: {e}")
                continue

        # Add trial results
        results_list.extend(trial_results)

    # Create DataFrame
    results_df = pd.DataFrame(results_list)

    if save_results and not results_df.empty:
        # Save raw results
        csv_path = os.path.join(output_dir, f"parameter_sweep_{timestamp}.csv")
        results_df.to_csv(csv_path, index=False)
        print(f"\n✓ Results saved to: {csv_path}")

        # Save aggregated results (mean across trials)
        agg_cols = ['distance_based', 'load', 'percent_max_dist', 'bundling',
                    'max_bundle_size', 'hours', 'peaks', 'sigma', 'grace_time']

        agg_df = results_df.groupby(agg_cols).agg({
            # Primary metrics
            'avg_distance': ['mean', 'std'],
            'avg_time': ['mean', 'std'],
            'total_vehicles': ['mean', 'std'],
            'success_rate': ['mean', 'std'],
            'completion_rate': ['mean', 'std'],

            # Percentile metrics
            'p90_time': ['mean', 'std'],
            'p95_time': ['mean', 'std'],
            'p99_time': ['mean', 'std'],
            'p90_distance': ['mean', 'std'],
            'p95_distance': ['mean', 'std'],

            # Other metrics
            'num_bundles': ['mean', 'std'],
            'deliveries_per_vehicle': ['mean', 'std'],
        }).reset_index()

        agg_csv_path = os.path.join(output_dir, f"parameter_sweep_aggregated_{timestamp}.csv")
        agg_df.to_csv(agg_csv_path, index=False)
        print(f"✓ Aggregated results saved to: {agg_csv_path}")

    return results_df


def compute_pareto_frontier(results_df, objectives=['avg_distance', 'avg_time', 'total_vehicles'],
                            minimize=True):
    """
    Compute Pareto frontier for multi-objective optimization.

    Args:
        results_df: DataFrame with results
        objectives: List of objective column names
        minimize: If True, minimize objectives; if False, maximize

    Returns:
        DataFrame with Pareto-optimal solutions
    """
    # Extract objective values
    obj_values = results_df[objectives].values

    # For maximization, negate the values
    if not minimize:
        obj_values = -obj_values

    # Find Pareto-optimal points
    is_pareto = np.ones(len(obj_values), dtype=bool)

    for i, point in enumerate(obj_values):
        if is_pareto[i]:
            # Check if any other point dominates this point
            is_pareto[is_pareto] = np.any(obj_values[is_pareto] < point, axis=1)
            is_pareto[i] = True  # Keep current point

    pareto_df = results_df[is_pareto].copy()
    pareto_df['is_pareto'] = True

    return pareto_df


def analyze_multi_objective_results(results_df, output_dir="./experiment_results/"):
    """
    Comprehensive multi-objective analysis with Pareto frontier and trade-offs.
    """

    if results_df.empty:
        print("No results to analyze!")
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Aggregate across trials
    agg_cols = ['distance_based', 'load', 'percent_max_dist', 'bundling', 'max_bundle_size']
    agg_df = results_df.groupby(agg_cols).agg({
        'avg_distance': 'mean',
        'avg_time': 'mean',
        'total_vehicles': 'mean',
        'success_rate': 'mean',
        'completion_rate': 'mean',
        'p90_time': 'mean',
        'p95_time': 'mean',
        'p99_time': 'mean',
        'p90_distance': 'mean',
        'p95_distance': 'mean',
        'num_bundles': 'mean',
        'deliveries_per_vehicle': 'mean',
    }).reset_index()

    # Compute Pareto frontier for the three main objectives
    pareto_df = compute_pareto_frontier(
        agg_df,
        objectives=['avg_distance', 'avg_time', 'total_vehicles'],
        minimize=True
    )

    print("\n" + "="*80)
    print(f"PARETO-OPTIMAL CONFIGURATIONS: {len(pareto_df)} out of {len(agg_df)}")
    print("="*80)
    print(pareto_df[agg_cols + ['avg_distance', 'avg_time', 'total_vehicles', 'success_rate']].to_string(index=False))

    # Save Pareto frontier
    pareto_csv = os.path.join(output_dir, f"pareto_frontier_{timestamp}.csv")
    pareto_df.to_csv(pareto_csv, index=False)
    print(f"\n✓ Pareto frontier saved to: {pareto_csv}")

    # Create comprehensive visualizations
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

    fig.suptitle('Multi-Objective Trade-off Analysis', fontsize=18, fontweight='bold')

    # === 1. 3D Pareto Frontier ===
    ax1 = fig.add_subplot(gs[0, :], projection='3d')

    # Plot all solutions
    scatter_all = ax1.scatter(
        agg_df['avg_distance'],
        agg_df['avg_time'],
        agg_df['total_vehicles'],
        c='lightgray', s=50, alpha=0.3, label='All Solutions'
    )

    # Plot Pareto-optimal solutions
    scatter_pareto = ax1.scatter(
        pareto_df['avg_distance'],
        pareto_df['avg_time'],
        pareto_df['total_vehicles'],
        c='red', s=200, marker='*', edgecolors='black', linewidths=2,
        label='Pareto-Optimal', zorder=10
    )

    ax1.set_xlabel('Avg Distance (m)', fontsize=11, labelpad=10)
    ax1.set_ylabel('Avg Time (s)', fontsize=11, labelpad=10)
    ax1.set_zlabel('Total Vehicles', fontsize=11, labelpad=10)
    ax1.set_title('3D Pareto Frontier: Distance vs Time vs Vehicles', fontsize=13, pad=20)
    ax1.legend(loc='upper left', fontsize=10)
    ax1.view_init(elev=20, azim=45)

    # === 2. Distance vs Time Trade-off ===
    ax2 = fig.add_subplot(gs[1, 0])

    for bundling in [True, False]:
        data = agg_df[agg_df['bundling'] == bundling]
        label = 'With Bundling' if bundling else 'Without Bundling'
        ax2.scatter(data['avg_distance'], data['avg_time'],
                   s=80, alpha=0.6, label=label)

    # Highlight Pareto frontier
    ax2.scatter(pareto_df['avg_distance'], pareto_df['avg_time'],
               s=200, c='red', marker='*', edgecolors='black', linewidths=2,
               label='Pareto-Optimal', zorder=10)

    ax2.set_xlabel('Avg Distance (m)', fontsize=11)
    ax2.set_ylabel('Avg Time (s)', fontsize=11)
    ax2.set_title('Trade-off: Distance vs Time', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    # === 3. Distance vs Vehicles Trade-off ===
    ax3 = fig.add_subplot(gs[1, 1])

    for bundling in [True, False]:
        data = agg_df[agg_df['bundling'] == bundling]
        label = 'With Bundling' if bundling else 'Without Bundling'
        ax3.scatter(data['avg_distance'], data['total_vehicles'],
                   s=80, alpha=0.6, label=label)

    ax3.scatter(pareto_df['avg_distance'], pareto_df['total_vehicles'],
               s=200, c='red', marker='*', edgecolors='black', linewidths=2,
               label='Pareto-Optimal', zorder=10)

    ax3.set_xlabel('Avg Distance (m)', fontsize=11)
    ax3.set_ylabel('Total Vehicles', fontsize=11)
    ax3.set_title('Trade-off: Distance vs Vehicles', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)

    # === 4. Time vs Vehicles Trade-off ===
    ax4 = fig.add_subplot(gs[1, 2])

    for bundling in [True, False]:
        data = agg_df[agg_df['bundling'] == bundling]
        label = 'With Bundling' if bundling else 'Without Bundling'
        ax4.scatter(data['avg_time'], data['total_vehicles'],
                   s=80, alpha=0.6, label=label)

    ax4.scatter(pareto_df['avg_time'], pareto_df['total_vehicles'],
               s=200, c='red', marker='*', edgecolors='black', linewidths=2,
               label='Pareto-Optimal', zorder=10)

    ax4.set_xlabel('Avg Time (s)', fontsize=11)
    ax4.set_ylabel('Total Vehicles', fontsize=11)
    ax4.set_title('Trade-off: Time vs Vehicles', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3)

    # === 5. Percentile Analysis: Time Distribution ===
    ax5 = fig.add_subplot(gs[2, 0])

    time_metrics = ['avg_time', 'p90_time', 'p95_time', 'p99_time']
    x_pos = np.arange(len(time_metrics))
    width = 0.35

    bundling_data = agg_df[agg_df['bundling'] == True][time_metrics].mean()
    no_bundling_data = agg_df[agg_df['bundling'] == False][time_metrics].mean()

    ax5.bar(x_pos - width/2, bundling_data, width, label='With Bundling', alpha=0.8)
    ax5.bar(x_pos + width/2, no_bundling_data, width, label='Without Bundling', alpha=0.8)

    ax5.set_xlabel('Time Metric', fontsize=11)
    ax5.set_ylabel('Time (seconds)', fontsize=11)
    ax5.set_title('Time Percentile Analysis', fontsize=12, fontweight='bold')
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels(['Mean', 'P90', 'P95', 'P99'], fontsize=10)
    ax5.legend(fontsize=9)
    ax5.grid(True, alpha=0.3, axis='y')

    # === 6. Percentile Analysis: Distance Distribution ===
    ax6 = fig.add_subplot(gs[2, 1])

    dist_metrics = ['avg_distance', 'p90_distance', 'p95_distance']
    x_pos = np.arange(len(dist_metrics))

    bundling_data = agg_df[agg_df['bundling'] == True][dist_metrics].mean()
    no_bundling_data = agg_df[agg_df['bundling'] == False][dist_metrics].mean()

    ax6.bar(x_pos - width/2, bundling_data, width, label='With Bundling', alpha=0.8)
    ax6.bar(x_pos + width/2, no_bundling_data, width, label='Without Bundling', alpha=0.8)

    ax6.set_xlabel('Distance Metric', fontsize=11)
    ax6.set_ylabel('Distance (meters)', fontsize=11)
    ax6.set_title('Distance Percentile Analysis', fontsize=12, fontweight='bold')
    ax6.set_xticks(x_pos)
    ax6.set_xticklabels(['Mean', 'P90', 'P95'], fontsize=10)
    ax6.legend(fontsize=9)
    ax6.grid(True, alpha=0.3, axis='y')

    # === 7. Success Rate vs Efficiency ===
    ax7 = fig.add_subplot(gs[2, 2])

    # Calculate efficiency score (lower is better)
    agg_df['efficiency_score'] = (
        agg_df['avg_distance'] / agg_df['avg_distance'].max() +
        agg_df['avg_time'] / agg_df['avg_time'].max() +
        agg_df['total_vehicles'] / agg_df['total_vehicles'].max()
    ) / 3

    for bundling in [True, False]:
        data = agg_df[agg_df['bundling'] == bundling]
        label = 'With Bundling' if bundling else 'Without Bundling'
        ax7.scatter(data['efficiency_score'], data['success_rate'],
                   s=80, alpha=0.6, label=label)

    ax7.set_xlabel('Efficiency Score (lower is better)', fontsize=11)
    ax7.set_ylabel('Success Rate', fontsize=11)
    ax7.set_title('Success Rate vs Overall Efficiency', fontsize=12, fontweight='bold')
    ax7.legend(fontsize=9)
    ax7.grid(True, alpha=0.3)

    # === 8. Heatmap: Distance by Load and Max Bundle Size ===
    ax8 = fig.add_subplot(gs[3, 0])

    bundling_data = agg_df[agg_df['bundling'] == True]
    if not bundling_data.empty:
        pivot = bundling_data.pivot_table(
            values='avg_distance',
            index='load',
            columns='max_bundle_size',
            aggfunc='mean'
        )
        sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd_r', ax=ax8, cbar_kws={'label': 'Avg Distance (m)'})
        ax8.set_title('Avg Distance: Load vs Max Bundle Size', fontsize=12, fontweight='bold')
        ax8.set_xlabel('Max Bundle Size', fontsize=11)
        ax8.set_ylabel('Load', fontsize=11)

    # === 9. Heatmap: Time by Load and Max Bundle Size ===
    ax9 = fig.add_subplot(gs[3, 1])

    if not bundling_data.empty:
        pivot = bundling_data.pivot_table(
            values='avg_time',
            index='load',
            columns='max_bundle_size',
            aggfunc='mean'
        )
        sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax9, cbar_kws={'label': 'Avg Time (s)'})
        ax9.set_title('Avg Time: Load vs Max Bundle Size', fontsize=12, fontweight='bold')
        ax9.set_xlabel('Max Bundle Size', fontsize=11)
        ax9.set_ylabel('Load', fontsize=11)

    # === 10. Heatmap: Vehicles by Load and Max Bundle Size ===
    ax10 = fig.add_subplot(gs[3, 2])

    if not bundling_data.empty:
        pivot = bundling_data.pivot_table(
            values='total_vehicles',
            index='load',
            columns='max_bundle_size',
            aggfunc='mean'
        )
        sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu_r', ax=ax10, cbar_kws={'label': 'Total Vehicles'})
        ax10.set_title('Total Vehicles: Load vs Max Bundle Size', fontsize=12, fontweight='bold')
        ax10.set_xlabel('Max Bundle Size', fontsize=11)
        ax10.set_ylabel('Load', fontsize=11)

    plt.tight_layout()

    # Save figure
    fig_path = os.path.join(output_dir, f"multi_objective_analysis_{timestamp}.png")
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"✓ Multi-objective analysis saved to: {fig_path}")
    plt.show()

    # === Print detailed trade-off analysis ===
    print("\n" + "="*80)
    print("TRADE-OFF ANALYSIS SUMMARY")
    print("="*80)

    # Compare bundling vs no bundling on all metrics
    bundling_avg = agg_df[agg_df['bundling'] == True][['avg_distance', 'avg_time', 'total_vehicles', 'success_rate']].mean()
    no_bundling_avg = agg_df[agg_df['bundling'] == False][['avg_distance', 'avg_time', 'total_vehicles', 'success_rate']].mean()

    print("\nAVERAGE PERFORMANCE:")
    print(f"                      With Bundling    Without Bundling    Difference")
    print(f"Avg Distance (m):     {bundling_avg['avg_distance']:>10.1f}    {no_bundling_avg['avg_distance']:>10.1f}    {bundling_avg['avg_distance'] - no_bundling_avg['avg_distance']:>10.1f} ({(bundling_avg['avg_distance'] / no_bundling_avg['avg_distance'] - 1) * 100:+.1f}%)")
    print(f"Avg Time (s):         {bundling_avg['avg_time']:>10.1f}    {no_bundling_avg['avg_time']:>10.1f}    {bundling_avg['avg_time'] - no_bundling_avg['avg_time']:>10.1f} ({(bundling_avg['avg_time'] / no_bundling_avg['avg_time'] - 1) * 100:+.1f}%)")
    print(f"Total Vehicles:       {bundling_avg['total_vehicles']:>10.1f}    {no_bundling_avg['total_vehicles']:>10.1f}    {bundling_avg['total_vehicles'] - no_bundling_avg['total_vehicles']:>10.1f} ({(bundling_avg['total_vehicles'] / no_bundling_avg['total_vehicles'] - 1) * 100:+.1f}%)")
    print(f"Success Rate:         {bundling_avg['success_rate']:>10.3f}    {no_bundling_avg['success_rate']:>10.3f}    {bundling_avg['success_rate'] - no_bundling_avg['success_rate']:>10.3f} ({(bundling_avg['success_rate'] / no_bundling_avg['success_rate'] - 1) * 100:+.1f}%)")

    # Percentile comparison
    print("\n\nPERCENTILE METRICS (WITH BUNDLING):")
    bundling_percentiles = agg_df[agg_df['bundling'] == True][['avg_time', 'p90_time', 'p95_time', 'p99_time']].mean()
    print(f"Time - Mean: {bundling_percentiles['avg_time']:.1f}s, P90: {bundling_percentiles['p90_time']:.1f}s, P95: {bundling_percentiles['p95_time']:.1f}s, P99: {bundling_percentiles['p99_time']:.1f}s")

    bundling_dist_percentiles = agg_df[agg_df['bundling'] == True][['avg_distance', 'p90_distance', 'p95_distance']].mean()
    print(f"Distance - Mean: {bundling_dist_percentiles['avg_distance']:.1f}m, P90: {bundling_dist_percentiles['p90_distance']:.1f}m, P95: {bundling_dist_percentiles['p95_distance']:.1f}m")

    print("\nPERCENTILE METRICS (WITHOUT BUNDLING):")
    no_bundling_percentiles = agg_df[agg_df['bundling'] == False][['avg_time', 'p90_time', 'p95_time', 'p99_time']].mean()
    print(f"Time - Mean: {no_bundling_percentiles['avg_time']:.1f}s, P90: {no_bundling_percentiles['p90_time']:.1f}s, P95: {no_bundling_percentiles['p95_time']:.1f}s, P99: {no_bundling_percentiles['p99_time']:.1f}s")

    no_bundling_dist_percentiles = agg_df[agg_df['bundling'] == False][['avg_distance', 'p90_distance', 'p95_distance']].mean()
    print(f"Distance - Mean: {no_bundling_dist_percentiles['avg_distance']:.1f}m, P90: {no_bundling_dist_percentiles['p90_distance']:.1f}m, P95: {no_bundling_dist_percentiles['p95_distance']:.1f}m")

    # Best configurations for each objective
    print("\n" + "="*80)
    print("BEST CONFIGURATIONS FOR EACH OBJECTIVE:")
    print("="*80)

    best_distance = agg_df.loc[agg_df['avg_distance'].idxmin()]
    print(f"\nLOWEST AVG DISTANCE: {best_distance['avg_distance']:.1f}m")
    print(f"  Config: {best_distance['distance_based']}, load={best_distance['load']}, bundling={best_distance['bundling']}, max_bundle={best_distance['max_bundle_size']}")
    print(f"  Avg Time: {best_distance['avg_time']:.1f}s, Vehicles: {best_distance['total_vehicles']:.0f}, Success: {best_distance['success_rate']:.2%}")

    best_time = agg_df.loc[agg_df['avg_time'].idxmin()]
    print(f"\nLOWEST AVG TIME: {best_time['avg_time']:.1f}s")
    print(f"  Config: {best_time['distance_based']}, load={best_time['load']}, bundling={best_time['bundling']}, max_bundle={best_time['max_bundle_size']}")
    print(f"  Avg Distance: {best_time['avg_distance']:.1f}m, Vehicles: {best_time['total_vehicles']:.0f}, Success: {best_time['success_rate']:.2%}")

    best_vehicles = agg_df.loc[agg_df['total_vehicles'].idxmin()]
    print(f"\nLOWEST VEHICLE COUNT: {best_vehicles['total_vehicles']:.0f}")
    print(f"  Config: {best_vehicles['distance_based']}, load={best_vehicles['load']}, bundling={best_vehicles['bundling']}, max_bundle={best_vehicles['max_bundle_size']}")
    print(f"  Avg Distance: {best_vehicles['avg_distance']:.1f}m, Avg Time: {best_vehicles['avg_time']:.1f}s, Success: {best_vehicles['success_rate']:.2%}")

    best_success = agg_df.loc[agg_df['success_rate'].idxmax()]
    print(f"\nHIGHEST SUCCESS RATE: {best_success['success_rate']:.2%}")
    print(f"  Config: {best_success['distance_based']}, load={best_success['load']}, bundling={best_success['bundling']}, max_bundle={best_success['max_bundle_size']}")
    print(f"  Avg Distance: {best_success['avg_distance']:.1f}m, Avg Time: {best_success['avg_time']:.1f}s, Vehicles: {best_success['total_vehicles']:.0f}")

    # Balanced configuration (normalize and find minimum combined score)
    agg_df_normalized = agg_df.copy()
    agg_df_normalized['norm_distance'] = (agg_df['avg_distance'] - agg_df['avg_distance'].min()) / (agg_df['avg_distance'].max() - agg_df['avg_distance'].min())
    agg_df_normalized['norm_time'] = (agg_df['avg_time'] - agg_df['avg_time'].min()) / (agg_df['avg_time'].max() - agg_df['avg_time'].min())
    agg_df_normalized['norm_vehicles'] = (agg_df['total_vehicles'] - agg_df['total_vehicles'].min()) / (agg_df['total_vehicles'].max() - agg_df['total_vehicles'].min())
    agg_df_normalized['combined_score'] = (agg_df_normalized['norm_distance'] + agg_df_normalized['norm_time'] + agg_df_normalized['norm_vehicles']) / 3

    best_balanced = agg_df.loc[agg_df_normalized['combined_score'].idxmin()]
    print(f"\nBEST BALANCED CONFIGURATION (equal weight on all 3 objectives):")
    print(f"  Config: {best_balanced['distance_based']}, load={best_balanced['load']}, bundling={best_balanced['bundling']}, max_bundle={best_balanced['max_bundle_size']}")
    print(f"  Avg Distance: {best_balanced['avg_distance']:.1f}m, Avg Time: {best_balanced['avg_time']:.1f}s, Vehicles: {best_balanced['total_vehicles']:.0f}, Success: {best_balanced['success_rate']:.2%}")

    return agg_df, pareto_df


def generate_pareto_trade_off_report(pareto_df, output_dir="./experiment_results/"):
    """
    Generate detailed report on Pareto-optimal solutions showing trade-offs.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("\n" + "="*100)
    print("DETAILED PARETO-OPTIMAL SOLUTIONS REPORT")
    print("="*100)

    # Sort by different objectives to show extremes
    pareto_sorted_by_distance = pareto_df.sort_values('avg_distance')
    pareto_sorted_by_time = pareto_df.sort_values('avg_time')
    pareto_sorted_by_vehicles = pareto_df.sort_values('total_vehicles')

    print("\n--- PARETO SOLUTIONS SORTED BY DISTANCE (LOW TO HIGH) ---")
    print(pareto_sorted_by_distance[['distance_based', 'load', 'bundling', 'max_bundle_size',
                                     'avg_distance', 'avg_time', 'total_vehicles', 'success_rate']].to_string(index=False))

    print("\n--- PARETO SOLUTIONS SORTED BY TIME (LOW TO HIGH) ---")
    print(pareto_sorted_by_time[['distance_based', 'load', 'bundling', 'max_bundle_size',
                                 'avg_distance', 'avg_time', 'total_vehicles', 'success_rate']].to_string(index=False))

    print("\n--- PARETO SOLUTIONS SORTED BY VEHICLES (LOW TO HIGH) ---")
    print(pareto_sorted_by_vehicles[['distance_based', 'load', 'bundling', 'max_bundle_size',
                                     'avg_distance', 'avg_time', 'total_vehicles', 'success_rate']].to_string(index=False))

    # Analyze trade-offs between consecutive Pareto points
    print("\n" + "="*100)
    print("TRADE-OFF ANALYSIS BETWEEN CONSECUTIVE PARETO POINTS")
    print("="*100)

    pareto_sorted = pareto_df.sort_values(['avg_distance', 'avg_time', 'total_vehicles'])

    for i in range(len(pareto_sorted) - 1):
        current = pareto_sorted.iloc[i]
        next_point = pareto_sorted.iloc[i + 1]

        dist_change = next_point['avg_distance'] - current['avg_distance']
        time_change = next_point['avg_time'] - current['avg_time']
        vehicle_change = next_point['total_vehicles'] - current['total_vehicles']

        print(f"\nTRADE-OFF {i+1}: Moving from Config {i+1} to Config {i+2}")
        print(f"  Distance: {current['avg_distance']:.1f}m → {next_point['avg_distance']:.1f}m (Δ={dist_change:+.1f}m, {(dist_change/current['avg_distance']*100):+.1f}%)")
        print(f"  Time:     {current['avg_time']:.1f}s → {next_point['avg_time']:.1f}s (Δ={time_change:+.1f}s, {(time_change/current['avg_time']*100):+.1f}%)")
        print(f"  Vehicles: {current['total_vehicles']:.0f} → {next_point['total_vehicles']:.0f} (Δ={vehicle_change:+.0f}, {(vehicle_change/current['total_vehicles']*100):+.1f}%)")
        print(f"  Config: [{current['distance_based']}, load={current['load']:.0f}, bundle={current['bundling']}, size={current['max_bundle_size']:.0f}] → [{next_point['distance_based']}, load={next_point['load']:.0f}, bundle={next_point['bundling']}, size={next_point['max_bundle_size']:.0f}]")

    # Create trade-off visualization specifically for Pareto frontier
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Pareto Frontier Trade-off Details', fontsize=16, fontweight='bold')

    # 1. Distance vs Time on Pareto frontier with labels
    ax = axes[0, 0]
    ax.plot(pareto_sorted['avg_distance'], pareto_sorted['avg_time'],
            'ro-', linewidth=2, markersize=10, label='Pareto Frontier')

    for idx, row in pareto_sorted.iterrows():
        label = f"L{int(row['load'])}\nB{int(row['max_bundle_size']) if row['bundling'] else 'N'}"
        ax.annotate(label, (row['avg_distance'], row['avg_time']),
                   fontsize=8, ha='center', va='bottom')

    ax.set_xlabel('Average Distance (m)', fontsize=12)
    ax.set_ylabel('Average Time (s)', fontsize=12)
    ax.set_title('Pareto Frontier: Distance vs Time', fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend()

    # 2. Distance vs Vehicles
    ax = axes[0, 1]
    ax.plot(pareto_sorted['avg_distance'], pareto_sorted['total_vehicles'],
            'go-', linewidth=2, markersize=10, label='Pareto Frontier')

    for idx, row in pareto_sorted.iterrows():
        label = f"L{int(row['load'])}\nB{int(row['max_bundle_size']) if row['bundling'] else 'N'}"
        ax.annotate(label, (row['avg_distance'], row['total_vehicles']),
                   fontsize=8, ha='center', va='bottom')

    ax.set_xlabel('Average Distance (m)', fontsize=12)
    ax.set_ylabel('Total Vehicles', fontsize=12)
    ax.set_title('Pareto Frontier: Distance vs Vehicles', fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend()

    # 3. Time vs Vehicles
    ax = axes[1, 0]
    ax.plot(pareto_sorted['avg_time'], pareto_sorted['total_vehicles'],
            'bo-', linewidth=2, markersize=10, label='Pareto Frontier')

    for idx, row in pareto_sorted.iterrows():
        label = f"L{int(row['load'])}\nB{int(row['max_bundle_size']) if row['bundling'] else 'N'}"
        ax.annotate(label, (row['avg_time'], row['total_vehicles']),
                   fontsize=8, ha='center', va='bottom')

    ax.set_xlabel('Average Time (s)', fontsize=12)
    ax.set_ylabel('Total Vehicles', fontsize=12)
    ax.set_title('Pareto Frontier: Time vs Vehicles', fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend()

    # 4. Radar chart comparing Pareto solutions
    ax = axes[1, 1]

    # Normalize metrics for radar chart
    categories = ['Distance\n(normalized)', 'Time\n(normalized)', 'Vehicles\n(normalized)', 'Success Rate']

    # Select up to 5 Pareto solutions to display
    num_solutions = min(5, len(pareto_sorted))
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle

    ax = plt.subplot(2, 2, 4, projection='polar')

    colors = plt.cm.viridis(np.linspace(0, 1, num_solutions))

    for i in range(num_solutions):
        row = pareto_sorted.iloc[i]

        # Normalize (invert distance, time, vehicles so higher is better for radar chart)
        norm_dist = 1 - (row['avg_distance'] - pareto_sorted['avg_distance'].min()) / (pareto_sorted['avg_distance'].max() - pareto_sorted['avg_distance'].min() + 1e-6)
        norm_time = 1 - (row['avg_time'] - pareto_sorted['avg_time'].min()) / (pareto_sorted['avg_time'].max() - pareto_sorted['avg_time'].min() + 1e-6)
        norm_vehicles = 1 - (row['total_vehicles'] - pareto_sorted['total_vehicles'].min()) / (pareto_sorted['total_vehicles'].max() - pareto_sorted['total_vehicles'].min() + 1e-6)

        values = [norm_dist, norm_time, norm_vehicles, row['success_rate']]
        values += values[:1]  # Complete the circle

        label = f"Config {i+1}: L={int(row['load'])}, B={'Y' if row['bundling'] else 'N'}"
        ax.plot(angles, values, 'o-', linewidth=2, color=colors[i], label=label)
        ax.fill(angles, values, alpha=0.15, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title('Pareto Solutions Comparison\n(Higher is Better)', fontsize=12, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=8)
    ax.grid(True)

    plt.tight_layout()

    fig_path = os.path.join(output_dir, f"pareto_tradeoffs_{timestamp}.png")
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"\n✓ Pareto trade-off visualization saved to: {fig_path}")
    plt.show()


def create_decision_support_table(pareto_df, output_dir="./experiment_results/"):
    """
    Create a decision support table to help choose configuration based on priorities.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("\n" + "="*100)
    print("DECISION SUPPORT: WHICH PARETO-OPTIMAL CONFIGURATION TO CHOOSE?")
    print("="*100)

    # Create priority scenarios
    scenarios = {
        "Cost-Focused (Minimize Distance & Vehicles)": {
            'weights': {'avg_distance': 0.5, 'avg_time': 0.1, 'total_vehicles': 0.4}
        },
        "Speed-Focused (Minimize Time)": {
            'weights': {'avg_distance': 0.2, 'avg_time': 0.6, 'total_vehicles': 0.2}
        },
        "Efficiency-Focused (Minimize Vehicles)": {
            'weights': {'avg_distance': 0.2, 'avg_time': 0.2, 'total_vehicles': 0.6}
        },
        "Balanced": {
            'weights': {'avg_distance': 0.33, 'avg_time': 0.33, 'total_vehicles': 0.34}
        },
        "Quality-Focused (Minimize Time & Distance)": {
            'weights': {'avg_distance': 0.45, 'avg_time': 0.45, 'total_vehicles': 0.1}
        }
    }

    decision_table = []

    for scenario_name, scenario in scenarios.items():
        # Normalize metrics (0-1 scale, lower is better)
        pareto_norm = pareto_df.copy()
        pareto_norm['norm_distance'] = (pareto_norm['avg_distance'] - pareto_norm['avg_distance'].min()) / (pareto_norm['avg_distance'].max() - pareto_norm['avg_distance'].min() + 1e-6)
        pareto_norm['norm_time'] = (pareto_norm['avg_time'] - pareto_norm['avg_time'].min()) / (pareto_norm['avg_time'].max() - pareto_norm['avg_time'].min() + 1e-6)
        pareto_norm['norm_vehicles'] = (pareto_norm['total_vehicles'] - pareto_norm['total_vehicles'].min()) / (pareto_norm['total_vehicles'].max() - pareto_norm['total_vehicles'].min() + 1e-6)

        # Calculate weighted score
        pareto_norm['weighted_score'] = (
            scenario['weights']['avg_distance'] * pareto_norm['norm_distance'] +
            scenario['weights']['avg_time'] * pareto_norm['norm_time'] +
            scenario['weights']['avg_vehicles'] * pareto_norm['norm_vehicles']
        )

        # Find best configuration for this scenario
        best_idx = pareto_norm['weighted_score'].idxmin()
        best_config = pareto_df.loc[best_idx]

        decision_table.append({
            'Scenario': scenario_name,
            'Distance Priority': f"{scenario['weights']['avg_distance']:.0%}",
            'Time Priority': f"{scenario['weights']['avg_time']:.0%}",
            'Vehicle Priority': f"{scenario['weights']['avg_vehicles']:.0%}",
            'Best Config': f"{best_config['distance_based']}, L={int(best_config['load'])}, Bundle={'Y' if best_config['bundling'] else 'N'}, Size={int(best_config['max_bundle_size'])}",
            'Avg Distance (m)': f"{best_config['avg_distance']:.1f}",
            'Avg Time (s)': f"{best_config['avg_time']:.1f}",
            'Total Vehicles': f"{best_config['total_vehicles']:.0f}",
            'Success Rate': f"{best_config['success_rate']:.2%}"
        })

    decision_df = pd.DataFrame(decision_table)
    print(decision_df.to_string(index=False))

    # Save decision table
    csv_path = os.path.join(output_dir, f"decision_support_{timestamp}.csv")
    decision_df.to_csv(csv_path, index=False)
    print(f"\n✓ Decision support table saved to: {csv_path}")

    return decision_df



In [ ]:

# ============================================================================
# MAIN EXECUTION EXAMPLE
# ============================================================================

if __name__ == "__main__":

    print("="*80)
    print("MULTI-OBJECTIVE DELIVERY OPTIMIZATION ANALYSIS")
    print("="*80)

    # Run parameter sweep
    results_df = run_parameter_sweep(
        distance_matrix=full_dist_matrix,
        time_matrix=full_time_matrix,
        num_hotspots=num_hotspots,
        census_gdf=census_gdf,

        # Define parameter ranges to test
        distance_based_options=["closer", "further"],
        load_options=[50, 100, 150],
        percent_max_dist_options=[80, 90, 100],
        bundling_options=[True, False],
        max_bundle_size_options=[2, 3, 4],

        # Multiple trials for statistical validity
        num_trials=3,
        save_results=True,
        output_dir=data_dir+ "/experiment_results/"
    )

    # Comprehensive multi-objective analysis
    agg_results, pareto_results = analyze_multi_objective_results(
        results_df,
        output_dir=data_dir+"/experiment_results/"
    )

    # Generate detailed Pareto trade-off report
    generate_pareto_trade_off_report(
        pareto_results,
        output_dir=data_dir+"/experiment_results/"
    )

    # Create decision support table
    decision_table = create_decision_support_table(
        pareto_results,
        output_dir=data_dir+"/experiment_results/"
    )

    print("\n" + "="*80)
    print("ANALYSIS COMPLETE!")
    print("="*80)
    print("All results, visualizations, and reports have been saved to ./experiment_results/")